# Counting Arnold's Meanders in Lean

### A Lean companion to *Counting Arnold's Meanders with Verified Algorithms*

In 1988 Vladimir Arnold asked: in how many ways can a river cross a straight road $n$ times? The
river comes in from the south, crosses the road $n$ times without ever crossing itself, and
leaves to the east. The answers start $1, 1, 1, 2, 3, 8, 14, 42, 81, 262, \ldots$ (OEIS
A005316). No formula is known.

The paper ([First Pair](https://firstpair.org/books/arnold-meanders/)) describes a Lean 4
formalization that defines these numbers precisely and proves two fast algorithms correct: a
search with a parity rule, and a transfer matrix. This notebook teaches Lean by working through
that formalization. It assumes no Lean, only the mathematics of the paper's first sections.

**How the notebook teaches.** Every Lean feature is explained once, in a text cell, before the
first code cell that uses it. After that it is used freely. Every definition and theorem
appears once, and the Lean code is the library's code: the notebook builder copies each
declaration from the source files in `Arnold/`, so what you run here is what the paper
describes. The index at the end lists every feature with the section that introduces it.

**Running it.** Install the kernel once with `notebook/kernel/install.sh`, which builds the
[Lean REPL](https://github.com/leanprover-community/repl) for this project and registers the
Jupyter kernel *Lean 4 (Arnold)*. Run `lake build` in the repository first, so that Mathlib and
the library are compiled. The first cell imports Mathlib and takes up to a minute; the others
take a second or two each.

**How the cells fit together.** A Lean program is a file, read from top to bottom. Here the
file is cut into cells. Each cell continues from the cell run before it, as if they were one
file. Running a cell again replays it from the state it started in, so you can edit a
definition and run it again. A cell that begins with `import` starts a new file.

## Contents

1. Lean as a calculator
2. The road and the river: crossing chords
3. A river is a permutation
4. First theorems
5. Closed meanders
6. Closed meanders are open meanders
7. A search with a parity rule
8. Fast code that provably computes the same thing
9. The search is correct
10. The transfer matrix
11. Merging states is harmless
12. Using the whole library
13. Index of Lean features

## 1. Lean as a calculator

**`import`.** A Lean file begins by importing the modules it builds on. *Mathlib* is Lean's
library of formalized mathematics: numbers, finite sets, permutations, sums, and hundreds of
thousands of theorems about them. `import Mathlib` loads all of it. The library's own first
file, `Arnold/Defs.lean`, imports nothing, because it must compile into a fast program; we
import Mathlib at once, since we need it a few cells later.

In [1]:
import Mathlib

**`#eval`, `#check` and comments.** Commands that start with `#` ask Lean a question instead of
adding to the file. `#eval e` evaluates the expression `e` and prints the value. `#check e`
prints the *type* of `e`: every Lean expression has exactly one type, and Lean checks it before
doing anything else. Text after `--` up to the end of the line is a comment.

**Numbers and functions.** `Nat` is the type of natural numbers $0, 1, 2, \ldots$ Mathlib writes
it `ℕ` (type `\N` in an editor); the two names mean the same type. A numeral like `3` can stand
for a number of many types, and `(3 : ℕ)` says which one: this is a *type ascription*. A
function is applied by writing its argument after it, with a space and no parentheses:
`Nat.succ 4` is the successor of 4, and `max 3 7` applies `max` to two arguments. Subtraction on
`ℕ` stops at zero (`2 - 5 = 0`), `/` rounds down, and `%` is the remainder. `(a, b)` is the
pair of `a` and `b`; its type is written `A × B`. The arithmetic rules matter below,
because the formalization does a lot of arithmetic with bridge numbers.

In [2]:
#eval 2 + 3 * 4          -- 14
#eval (2 : ℕ) - 5        -- natural numbers stop at 0
#eval (17 / 5, 17 % 5)   -- quotient and remainder
#eval max 3 7
#check Nat.succ 4        -- the type of the expression, ℕ
#check (2 : ℕ) + 2 = 4   -- a statement is an expression too; its type is Prop

14


0


(3, 2)


7


Nat.succ 4 : ℕ


2 + 2 = 4 : Prop


The last line shows the idea that the whole formalization rests on. `2 + 2 = 4` is an
expression whose type is `Prop`, the type of *propositions*. A proposition is a statement that
may or may not be true. To prove it is to build an expression whose type is that proposition, and
Lean checks such proofs exactly as it checks that `2 + 3 * 4` is a number.

## 2. The road and the river: crossing chords

Draw the road as a horizontal line and number its bridges $0, 1, \ldots, n-1$ from west to
east. Between two consecutive crossings the river runs along an *arc* in one half-plane, below
the road or above it, and the arcs alternate sides. Two arcs on the same side are like two
chords of a disk: they cross exactly when their endpoints alternate along the road. So a river
is a meander exactly when no two arcs on the same side alternate. The first definition says
when two chords alternate.

**`namespace`.** Every name in the library lives in the namespace `Arnold`, so its full name is
`Arnold.Interleave`. `namespace Arnold` opens the namespace: until the matching `end Arnold`,
new names get the prefix automatically, and names inside it can be used without the prefix. The
namespace stays open across the next cells, as it would in one file.

**`def` and docstrings.** `def name (x y : T) : R := body` defines `name` as a function of
parameters `x` and `y` of type `T`, returning a value of type `R`. A comment between `/--` and
`-/` placed before a declaration is its *docstring*; Lean attaches it to the name, and editors
show it on hover.

**A definition whose value is a proposition.** `Interleave` returns a `Prop`: it does not
compute an answer, it *states* something about four numbers. The logical symbols are `¬`
(not), `∧` (and), `↔` (if and only if) and `<`. Read the body as: "`c` lies strictly between
`a` and `b`" is *not equivalent* to "`d` lies strictly between `a` and `b`". In other words,
exactly one of `c`, `d` is inside the chord `{a, b}`.

In [3]:
namespace Arnold

/-- The chords `{a, b}` and `{c, d}` (with four distinct endpoints on a line) cross exactly
when one of `c`, `d` lies strictly between `a` and `b` and the other does not. -/
def Interleave (a b c d : Nat) : Prop :=
  ¬ ((min a b < c ∧ c < max a b) ↔ (min a b < d ∧ d < max a b))

**Type classes and `instance`.** Lean cannot evaluate an arbitrary proposition: most
propositions about all natural numbers have no algorithm that decides them. A proposition `p`
can be *decided* when there is a value of type `Decidable p`, an algorithm that returns either a
proof of `p` or a proof of `¬ p`. `Decidable` is a *type class*: a family of types whose values
Lean finds automatically when it needs them. An `instance` declaration supplies such a value.
The instance below tells Lean how to decide `Interleave a b c d` for any four numbers.

**Tactics: `by`, `unfold`, `infer_instance`.** Instead of writing a value directly, we can
write `by` followed by *tactics*, commands that build the value step by step. `unfold
Interleave` replaces `Interleave a b c d` by its definition. The goal then becomes "decide a
negated `↔` of conjunctions of `<`", and `infer_instance` finds that instance by combining the
library's instances for `¬`, `↔`, `∧` and `<` on `ℕ`. A `;` separates tactics on one line.

In [4]:
instance (a b c d : Nat) : Decidable (Interleave a b c d) := by
  unfold Interleave; infer_instance

Now `#eval` can evaluate the proposition: it runs the decision procedure and prints `true` or
`false`. The chords $\{0, 2\}$ and $\{1, 3\}$ alternate; $\{0, 3\}$ and $\{1, 2\}$ are nested.

In [5]:
#eval Interleave 0 2 1 3
#eval Interleave 0 3 1 2

true


false


### No two arcs on one side cross

A river visits boundary points $P(0), P(1), P(2), \ldots$ and arc $k$ joins $P(k)$ to
$P(k+1)$. Arcs $0, 2, 4, \ldots$ lie on one side of the road and arcs $1, 3, 5, \ldots$ on the
other, so arcs $j$ and $k$ are on the same side exactly when $j \bmod 2 = k \bmod 2$.

**Function types and quantifiers.** `Nat → Nat` is the type of functions from `ℕ` to `ℕ`; the
river's path `P` is such a function. `∀ k < L, Q k` says "for every `k` with `k < L`, `Q k`",
and the `∀` can be followed by more conditions. Between propositions, `→` means *implies*:
`j % 2 = k % 2 → ¬ ...` reads "if arcs `j` and `k` are on the same side, they do not cross".
(Implication and function type are the same arrow, and that is no accident: a proof of
`A → B` is a function that turns any proof of `A` into a proof of `B`.)

In [6]:
/-- Given the boundary points `P 0, P 1, …` visited by a river, with arc `k` joining `P k` to
`P (k+1)` and arcs alternating between the two half-planes: no two of the first `L` arcs on the
same side cross. -/
def NoCross (P : Nat → Nat) (L : Nat) : Prop :=
  ∀ k < L, ∀ j < k, j % 2 = k % 2 → ¬ Interleave (P j) (P (j + 1)) (P k) (P (k + 1))

In [7]:
instance (P : Nat → Nat) (L : Nat) : Decidable (NoCross P L) := by
  unfold NoCross; infer_instance

**Anonymous functions, lists and dot notation.** `fun x => e` is the function that sends `x`
to `e`. `[0, 2, 1, 3]` is a *list*, a finite sequence; its type is `List ℕ`. Functions about a
type live in the namespace of the type, so the function that reads entry `k` of a list, or a
default value if the list is too short, is `List.getD`. *Dot notation* abbreviates
`List.getD l k 0` to `l.getD k 0`: Lean sees that `l` is a list and looks for `getD` in the
namespace `List`. We test
`NoCross` on two paths with four boundary points. The first, $0 \to 1 \to 2 \to 3$, has arcs
$\{0,1\}$ and $\{2,3\}$ on the same side, which do not cross. The second, $0 \to 2 \to 1 \to 3$,
has arcs $\{0,2\}$ and $\{1,3\}$ on the same side, which do.

In [8]:
#eval NoCross (fun k => k) 3
#eval NoCross (fun k => [0, 2, 1, 3].getD k 0) 3

true


false


## 3. A river is a permutation

A meander is determined by the order in which the river crosses the bridges: its $i$-th
crossing is at bridge $\sigma(i)$, where $\sigma$ is a permutation of $\{0, \ldots, n-1\}$. The
two ends of the river go off to infinity: it comes in from the south and leaves to the east.
Compactify each half-plane to a disk, so that the road plus the point at infinity is its
boundary. Then the two ends touch the boundary east of every bridge: the east end, which we call
point $n$, and the south end, point $n+1$. The path of the river is the sequence of boundary
points
$$n+1,\ \sigma(0),\ \sigma(1),\ \ldots,\ \sigma(n-1),\ n,$$
and arc $k$ joins the $k$-th and $(k+1)$-st of them. The river starts in the south, so the even
arcs are below the road and the odd arcs above.

**`Fin n` and permutations.** `Fin n` is the type of natural numbers less than `n`. An element
of `Fin n` is a pair of a number `i` and a proof of `i < n`. `Equiv.Perm (Fin n)` is the type of
permutations of `Fin n`: bijections from `Fin n` to itself. A permutation can be applied like a
function, `σ i`.

**Implicit arguments.** In `pathPt {n : ℕ} (σ : Equiv.Perm (Fin n)) (k : ℕ)` the braces make
`n` *implicit*. We write `pathPt σ k`, and Lean works `n` out from the type of `σ`.

**Anonymous constructors and dependent `if`.** `⟨a, b⟩` builds a value of a structure type
from its fields, when Lean can tell the type from context. Here `⟨k - 1, h⟩ : Fin n` is the
number `k - 1` with the proof `h : k - 1 < n`. That proof comes from the test itself: in
`if h : c then t else e`, the branch `t` may use `h : c`, and `e` may use `h : ¬ c`. This is a
*dependent* `if`; the plain `if c then t else e` does not name the fact.

**Coercions.** `(σ ⟨k - 1, h⟩ : ℕ)` asks for the result as a natural number. The result is an
element of `Fin n`, and Lean inserts the *coercion* that forgets the proof and keeps the
number. Lean prints a coercion as `↑x`.

In [9]:
/-- The `k`-th boundary point on the river's path (`k = 0, …, n+1`): first the south end
`n+1`, then the bridges in the order the river crosses them, then the east end `n`. -/
def pathPt {n : ℕ} (σ : Equiv.Perm (Fin n)) (k : ℕ) : ℕ :=
  if k = 0 then n + 1 else if h : k - 1 < n then (σ ⟨k - 1, h⟩ : ℕ) else n

The straight river that crosses bridges $0$ and $1$ in order visits $3, 0, 1, 2$. Here
`Equiv.refl` is the identity permutation, and `List.range 4` is the list `[0, 1, 2, 3]`.
`l.map f` applies the function `f` to every entry of the list `l`. A function of two
arguments given only its first, like `pathPt (Equiv.refl (Fin 2))`, is a function of the
remaining one.

In [10]:
#eval (List.range 4).map (pathPt (Equiv.refl (Fin 2)))

[3, 0, 1, 2]


The definition of a meander is now one line, and the decision procedure comes from the one for
`NoCross`, exactly as before.

In [11]:
/-- `σ` describes an open meander when no two arcs in the same half-plane cross.
Arc `k` (`k = 0, …, n`) joins `pathPt σ k` to `pathPt σ (k+1)`. -/
def IsMeander {n : ℕ} (σ : Equiv.Perm (Fin n)) : Prop :=
  NoCross (pathPt σ) (n + 1)

In [12]:
instance {n : ℕ} (σ : Equiv.Perm (Fin n)) : Decidable (IsMeander σ) := by
  unfold IsMeander; infer_instance

### Arnold's number

**Subtypes and `Fintype.card`.** `{σ : Equiv.Perm (Fin n) // IsMeander σ}` is the *subtype* of
permutations that are meanders. Its elements are pairs of a permutation and a proof that it is
a meander. A type with finitely many elements has a `Fintype` instance, and `Fintype.card T` is
the number of its elements. So the following is the definition of Arnold's number, word for
word: the number of permutations that describe a meander.

In [13]:
/-- **Arnold's number**: the number of ways a river can cross a straight road `n` times. -/
def openMeanderCount (n : ℕ) : ℕ :=
  Fintype.card {σ : Equiv.Perm (Fin n) // IsMeander σ}

This definition is the *specification*. It is easy to check against the problem, and every
faster algorithm in the library is proved equal to it. It can also be run, though only for
small $n$, because it tests all $n!$ permutations. The first seven values are
$1, 1, 1, 2, 3, 8, 14$.

In [14]:
#eval (List.range 7).map openMeanderCount

[1, 1, 1, 2, 3, 8, 14]


## 4. First theorems

The first facts in the library are that every $n$ has at least one meander and at most $n!$.
For at least one, take the river that zig-zags straight east across bridges
$0, 1, \ldots, n-1$. Its path is $n+1, 0, 1, \ldots, n-1, n$. Arc $0$ joins $n+1$ to $0$ and
encloses every later point; every other arc joins two neighbouring points, with nothing
strictly between them. So no two arcs alternate.

**`theorem` and `example`.** `theorem name (x : T) : P := proof` declares that `proof` proves
the proposition `P`, for every `x`. Lean checks that `proof` has type `P`, exactly as it checks
the value of a `def`. `example : P := proof` is a theorem without a name: Lean checks it and
then forgets it. We use examples to try out tactics before the real proofs.

**The goal and the first tactics.** Inside `by`, Lean shows a *goal*: hypotheses above a line,
the statement to prove below. Each tactic changes the goal until nothing is left.

* `intro x h` moves a `∀ x` or an `A →` from the goal into the hypotheses, naming them. It
  unfolds definitions if needed to find the `∀`.
* `rw [h]`, for `h : a = b`, rewrites `a` to `b` in the goal. `rw [h x y]` first fills in the
  arguments of a lemma `h`.
* `omega` proves goals in linear arithmetic over `ℕ` and `ℤ`: `+`, `-`, `*` by constants, `/`
  and `%` by constants, `<`, `≤`, `=`, and the logic connecting them. It is the workhorse of
  the library. `a ≠ b` means `¬ (a = b)`.

In [15]:
example : ∀ a b : ℕ, a = b + 1 → a ≠ 0 := by
  intro a b h
  rw [h]
  omega

**Case splits.** `split_ifs with h` splits a goal containing `if c then … else …` into two
goals: one with `h : c` and the branch `then`, one with `h : ¬ c` and the branch `else`. The
indented `·` (a centred dot) starts the proof of one goal, and the next `·` the next goal.
`by_cases h : c` splits on any decidable `c` in the same way.

In [16]:
example (a : ℕ) : (if a = 0 then 1 else a) ≠ 0 := by
  split_ifs with h
  · omega
  · omega

**`simp`, `show … by`, `rfl`.** `simp only [l₁, l₂]` rewrites the goal with the given lemmas,
repeatedly, and closes it if it becomes trivially true. `↓reduceIte` is a *simproc*, a small
procedure for `simp`: it replaces `if c then a else b` by `a` when it can prove `c`, or by `b`
when it can prove `¬ c`. A proof written inline as `show P by tac` hands `simp` the fact `P`.
`simp [l₁]` without `only` also uses the hundreds of lemmas Mathlib marks as simplification
rules. `rfl` proves `a = a`, and more generally `a = b` whenever both sides compute to the same
value. `unfold f` replaces `f` by its definition, as in Section 2.

In [17]:
example (m : ℕ) (h : 1 ≤ m) : (if m = 0 then 5 else m) = m := by
  simp only [show m ≠ 0 by omega, ↓reduceIte]

**`have`.** `have name : P := by …` proves an intermediate fact `P` and adds it to the
hypotheses. A lemma with side conditions takes their proofs as arguments, and `by` may appear
anywhere a term is expected, so `hpt k (by omega) (by omega)` proves the two side conditions of
`hpt` on the spot. Below, `hpt` computes every point of the straight river at once: point `m` is
`m - 1` for $1 \le m \le n+1$. The proof then reduces the claim to arithmetic about these
points, which `omega` finishes.

In [18]:
/-- The river that zig-zags straight east across bridges `0, 1, …, n-1` is a meander. -/
theorem isMeander_refl (n : ℕ) : IsMeander (Equiv.refl (Fin n)) := by
  intro k hk j hjk hpar
  have hpt : ∀ m : ℕ, 1 ≤ m → m ≤ n + 1 →
      pathPt (Equiv.refl (Fin n)) m = m - 1 := by
    intro m h1 h2
    unfold pathPt
    simp only [show m ≠ 0 by omega, ↓reduceIte]
    split_ifs with h
    · rfl
    · omega
  -- `k ≥ 2`, so arc `k` is the arc `(k-1, k)` between neighbouring points.
  have hk2 : 2 ≤ k := by omega
  rw [hpt (k + 1) (by omega) (by omega), hpt k (by omega) (by omega)]
  unfold Interleave
  by_cases hj0 : j = 0
  · -- Arc 0 is `(n+1, 0)`. It encloses every other lower arc.
    have h0 : pathPt (Equiv.refl (Fin n)) j = n + 1 := by simp [pathPt, hj0]
    rw [h0, hpt (j + 1) (by omega) (by omega)]
    omega
  · -- Arc `j` is `(j-1, j)`. No point lies strictly between its endpoints.
    rw [hpt j (by omega) (by omega), hpt (j + 1) (by omega) (by omega)]
    omega

**Proofs as terms.** A proof need not use tactics. `openMeanderCount_pos` is proved by a
single expression. `Fintype.card_pos_iff` is the Mathlib lemma `0 < Fintype.card α ↔ Nonempty α`.
For a proof `h : A ↔ B`, `h.mp : A → B` and `h.mpr : B → A` are its two directions. `Nonempty α`
is proved by an anonymous constructor holding an element of `α`, and the element of the subtype
is itself the pair `⟨σ, proof that σ is a meander⟩`. An underscore `_` asks Lean to fill in a
term it can infer, here the `n` in `Fin n`.

In [19]:
/-- For every `n`, a river can cross the road `n` times. -/
theorem openMeanderCount_pos (n : ℕ) : 0 < openMeanderCount n :=
  Fintype.card_pos_iff.mpr ⟨⟨Equiv.refl _, isMeander_refl n⟩⟩

**`calc`.** A `calc` block proves a chain of relations, one step per line. `_` at the start of a
line stands for the previous right-hand side, and the first `_` for the left side of the goal.
The bound $n!$ is the number of all permutations: a subtype has at most as many elements as the
whole type (`Fintype.card_subtype_le`), and there are $n!$ permutations of `Fin n`. Note
`n.factorial`: dot notation works on any value, here `Nat.factorial n`.

In [20]:
/-- There are at most `n!` meanders with `n` crossings, one per crossing order. -/
theorem openMeanderCount_le_factorial (n : ℕ) : openMeanderCount n ≤ n.factorial := by
  unfold openMeanderCount
  calc _ ≤ Fintype.card (Equiv.Perm (Fin n)) := Fintype.card_subtype_le _
    _ = n.factorial := by rw [Fintype.card_perm, Fintype.card_fin]

**`decide`.** For a decidable proposition, the tactic `decide` runs the decision procedure
*inside Lean's kernel*, the small program that checks every proof. If it returns `true`, the
kernel has verified the proposition by computation. These four facts are checked by enumerating
all permutations.

In [21]:
theorem openMeanderCount_0 : openMeanderCount 0 = 1 := by decide
theorem openMeanderCount_1 : openMeanderCount 1 = 1 := by decide
theorem openMeanderCount_2 : openMeanderCount 2 = 1 := by decide
theorem openMeanderCount_3 : openMeanderCount 3 = 2 := by decide

## 5. Closed meanders

A *closed* meander is a closed river, a loop, crossing the road $2n$ times. To count each loop
once, fix where it starts and which way it goes: start at the westmost bridge $0$ and leave it
upward. The river then visits bridges $\sigma(0) = 0, \sigma(1), \ldots, \sigma(2n-1)$, and arc
$k$ joins $\sigma(k)$ to $\sigma(k+1)$, indices taken modulo $2n$, so the last arc returns to
bridge $0$. Even arcs are above the road and odd arcs below.

**Proofs as arguments.** To evaluate $\sigma$ at position $k \bmod 2n$ we need an element of
`Fin (2 * n)`, so a proof that $k \bmod 2n < 2n$. The Mathlib lemma `Nat.mod_lt` is a function:
given `x` and a proof of `0 < y`, it returns a proof of `x % y < y`. We apply it like any
function, with `_` for the `x` that Lean can infer.

In [22]:
/-- The `k`-th bridge (indices taken mod `2n`) visited by a closed river. -/
def closedPt {n : ℕ} (σ : Equiv.Perm (Fin (2 * n))) (k : ℕ) : ℕ :=
  if h : 0 < 2 * n then (σ ⟨k % (2 * n), Nat.mod_lt _ h⟩ : ℕ) else 0

**Quantifying over proofs.** The normalization "start at bridge 0" says `σ ⟨0, h⟩ = ⟨0, h⟩`,
which only makes sense when $0 < 2n$. `∀ h : 0 < 2 * n, …` says: for every proof `h` of
`0 < 2 * n`, the following holds. For $n = 0$ there is no such proof, and the condition is
empty.

In [23]:
/-- A **closed meander** of order `n` is a closed river crossing the road `2n` times. To count
each curve once, fix where it starts and which way it goes: start at the westmost bridge `0`
and leave it into the upper half-plane. The river then visits bridges `σ 0 = 0, σ 1, …`, and
arc `k` joins `σ k` to `σ (k+1 mod 2n)`. Even arcs are upper and odd arcs are lower. -/
def IsClosedMeander {n : ℕ} (σ : Equiv.Perm (Fin (2 * n))) : Prop :=
  (∀ h : 0 < 2 * n, σ ⟨0, h⟩ = ⟨0, h⟩) ∧ NoCross (closedPt σ) (2 * n)

In [24]:
instance {n : ℕ} (σ : Equiv.Perm (Fin (2 * n))) : Decidable (IsClosedMeander σ) := by
  unfold IsClosedMeander; infer_instance

In [25]:
/-- The number of closed meanders of order `n` (OEIS A005315). -/
def closedMeanderCount (n : ℕ) : ℕ :=
  Fintype.card {σ : Equiv.Perm (Fin (2 * n)) // IsClosedMeander σ}

The closed counts start $1, 2, 8$ for $n = 1, 2, 3$. The next section proves the classical fact
behind the second line: a closed meander of order $n$ is an open meander with $2n - 1$
crossings.

In [26]:
#eval (closedMeanderCount 1, closedMeanderCount 2, closedMeanderCount 3)
#eval (openMeanderCount 1, openMeanderCount 3, openMeanderCount 5)

(1, 2, 8)


(1, 2, 8)


## 6. Closed meanders are open meanders

Take a closed meander with $2n$ crossings, started at bridge $0$. Delete that bridge: the two
arcs at it now run off to infinity in the west. Turn the picture through $180°$. The river now
comes in from infinity below the road, crosses it $2n - 1$ times, and leaves to the east: an
open meander. Every step can be undone, so the two kinds are in bijection. In Lean the
construction goes the other way, from an open meander $\tau$ with $2n+1$ crossings to a closed
one with $2n+2$.

**Equivalences.** `Equiv α β`, written `α ≃ β`, is a bijection packaged with its inverse. A
permutation is an `Equiv` from a type to itself. For `e : α ≃ β`, `e.symm : β ≃ α` is the
inverse, and for `f : β ≃ γ`, `e.trans f : α ≃ γ` is "first `e`, then `f`". `Fin.revPerm` is the
reversal $i \mapsto n - 1 - i$ of `Fin n`, the $180°$ turn. Mathlib's `Perm.decomposeFin` is an
equivalence between permutations of `Fin (n+1)` and pairs (where $0$ goes, a permutation of
`Fin n`). Its inverse, applied to `(0, τ.trans Fin.revPerm)`, builds the permutation that fixes
$0$ and follows the reversed $\tau$ on the rest.

**`open`.** `open Equiv` lets us write `Perm` for `Equiv.Perm`. Like `namespace`, it lasts until
the end of the enclosing namespace.

In [27]:
open Equiv

/-- Rotate an open meander with `2n+1` crossings by 180° and close it up through a new
westmost bridge `0`. -/
def closeUp {n : ℕ} (τ : Perm (Fin (2 * n + 1))) : Perm (Fin (2 * n + 1 + 1)) :=
  Perm.decomposeFin.symm (0, τ.trans Fin.revPerm)

**Proofs of `∀` are functions.** A proof of `∀ x, P x` is a function that takes `x` and returns
a proof of `P x`, so it can be written with `fun`. The next theorem says that `NoCross` holds
for one path exactly when it holds for another, provided the two paths agree on which pairs of
arcs alternate. The proof wraps the hypothesis `h` in Mathlib's *congruence lemmas*:
`forall₂_congr` says that if two statements are equivalent for every `x` and `y`, then so are
`∀ x y, …`; `imp_congr_right` and `not_congr` do the same for `→` and `¬`.

In [28]:
/-- `NoCross` only depends on whether each pair of same-side arcs crosses. -/
theorem noCross_congr {P Q : ℕ → ℕ} {L : ℕ}
    (h : ∀ k < L, ∀ j < k, j % 2 = k % 2 →
      (Interleave (P j) (P (j + 1)) (P k) (P (k + 1)) ↔
        Interleave (Q j) (Q (j + 1)) (Q k) (Q (k + 1)))) :
    NoCross P L ↔ NoCross Q L :=
  forall₂_congr fun k hk => forall₂_congr fun j hj =>
    imp_congr_right fun hp => not_congr (h k hk j hj hp)

**Disjunction and `this`.** `A ∨ B` means *A or B*. `pathPt_cases` says each point on the path
of an open meander with $2n+1$ crossings is the south end, a bridge, or the east end. `split_ifs
with h0 h1` names the facts for each nested `if` in turn. `have := e` without a name calls the
new fact `this`. `(σ i).isLt` is the proof stored in the `Fin` value `σ i` that its number is
below the bound.

In [29]:
/-- The boundary points of an open meander with `2n+1` crossings: the south end `2n+2`, then
bridges `≤ 2n`, then the east end `2n+1`. -/
theorem pathPt_cases {n : ℕ} (τ : Perm (Fin (2 * n + 1))) (i : ℕ) (hi : i ≤ 2 * n + 2) :
    (i = 0 ∧ pathPt τ i = 2 * n + 2) ∨ (1 ≤ i ∧ i ≤ 2 * n + 1 ∧ pathPt τ i ≤ 2 * n) ∨
      (i = 2 * n + 2 ∧ pathPt τ i = 2 * n + 1) := by
  unfold pathPt
  split_ifs with h0 h1
  · omega
  · have := (τ ⟨i - 1, h1⟩).isLt
    omega
  · omega

**More tactics.** The next proof says that after the turn, closed point $k$ is open point $k$
reflected by $x \mapsto 2n+1-x$.

* `closedPt (n := n + 1) …` passes the implicit argument `n` by name.
* `↓reduceDIte` is `↓reduceIte` for dependent `if`.
* `ext` proves two `Fin` values equal by proving their numbers equal; in general it reduces
  equality to equality of components or values.
* `rcases e with h | h` splits on a disjunction `e`, naming the fact in each case.
* `t <;> s` runs `s` on every goal that `t` produces.

In [30]:
/-- After the rotation, closed point `k` is the open point `k` reflected: `x ↦ 2n+1 - x`.
(Truncated subtraction sends the south end `2n+2` to the new bridge `0`.) -/
theorem closedPt_closeUp {n : ℕ} (τ : Perm (Fin (2 * n + 1))) (k : ℕ) (hk : k ≤ 2 * n + 2) :
    closedPt (n := n + 1) (closeUp τ) k = 2 * n + 1 - pathPt τ k := by
  unfold closedPt
  simp only [show 0 < 2 * (n + 1) by omega, ↓reduceDIte]
  by_cases hmid : 1 ≤ k ∧ k ≤ 2 * n + 1
  · have hidx : (⟨k % (2 * (n + 1)), Nat.mod_lt _ (by omega)⟩ : Fin (2 * n + 1 + 1)) =
        (⟨k - 1, by omega⟩ : Fin (2 * n + 1)).succ := by
      ext; simp only [Fin.val_succ]; rw [Nat.mod_eq_of_lt (by omega)]; omega
    rw [hidx]
    simp only [closeUp, Perm.decomposeFin_symm_apply_succ, swap_self, refl_apply,
      Equiv.trans_apply, Fin.revPerm_apply, Fin.val_succ, Fin.val_rev]
    unfold pathPt
    simp only [show k ≠ 0 by omega, show k - 1 < 2 * n + 1 by omega, ↓reduceIte, ↓reduceDIte]
    omega
  · have hidx : (⟨k % (2 * (n + 1)), Nat.mod_lt _ (by omega)⟩ : Fin (2 * n + 1 + 1)) = 0 := by
      ext; simp only [Fin.val_zero]
      rcases Nat.eq_zero_or_pos k with h | h
      · simp [h]
      · rw [show k = 2 * (n + 1) by omega, Nat.mod_self]
    rw [hidx]
    simp only [closeUp, Perm.decomposeFin_symm_apply_zero, Fin.val_zero]
    rcases pathPt_cases τ k hk with h | h | h <;> omega

**`refine` and holes; `generalize`.** `refine e` proves the goal with an expression `e` that may
contain holes `?_`. Each hole becomes a new goal. Below, the hole is the body of a `fun`: for
each pair of arcs, show that they alternate before the turn exactly when they alternate after
it. `generalize pathPt τ j = a at *` replaces the expression by a fresh variable `a`
everywhere. That leaves `omega` a problem in plain variables, which it solves, using the
`pathPt_cases` facts for each point.

In [31]:
/-- The rotated, closed-up river is a closed meander exactly when the original river is an open
meander. -/
theorem noCross_closeUp_iff {n : ℕ} (τ : Perm (Fin (2 * n + 1))) :
    NoCross (closedPt (n := n + 1) (closeUp τ)) (2 * (n + 1)) ↔ IsMeander τ := by
  unfold IsMeander
  refine noCross_congr fun k hk j hjk hpar => ?_
  rw [closedPt_closeUp τ j (by omega), closedPt_closeUp τ (j + 1) (by omega),
    closedPt_closeUp τ k (by omega), closedPt_closeUp τ (k + 1) (by omega)]
  have h1 := pathPt_cases τ j (by omega)
  have h2 := pathPt_cases τ (j + 1) (by omega)
  have h3 := pathPt_cases τ k (by omega)
  have h4 := pathPt_cases τ (k + 1) (by omega)
  generalize pathPt τ j = a at *
  generalize pathPt τ (j + 1) = b at *
  generalize pathPt τ k = c at *
  generalize pathPt τ (k + 1) = d at *
  unfold Interleave
  omega

In [32]:
theorem closeUp_zero {n : ℕ} (τ : Perm (Fin (2 * n + 1))) : closeUp τ 0 = 0 := by
  simp [closeUp]

**Injectivity, `exact` and `simpa`.** `exact e` closes the goal with the term `e`, whose type
must be the goal. `Function.Injective f` means $f(a) = f(b) \to a = b$. If
`h : a = b`, then `congrArg f h : f a = f b`. `simpa [l] using e` simplifies both the goal and
the term `e`, then closes the goal with `e`. `Equiv.ext` proves two equivalences equal by
proving they agree at every point.

In [33]:
theorem closeUp_injective {n : ℕ} : Function.Injective (closeUp (n := n)) := by
  intro τ τ' h
  have h' : τ.trans Fin.revPerm = τ'.trans Fin.revPerm := by
    have := congrArg (fun σ => (Perm.decomposeFin σ).2) h
    simpa [closeUp] using this
  exact Equiv.ext fun i => by simpa using congrArg (fun e => e i) h'

**Existentials and `obtain`.** `∃ x, P x` says that some `x` satisfies `P`. It is proved by an
anonymous constructor `⟨x, proof⟩`, and used by `obtain ⟨x, hx⟩ := h`, which takes the witness
and its property out of `h`. Patterns nest. The pattern `rfl` means: the fact here is an
equation `v = e`, so replace `v` by `e` everywhere. `subst h` does the same for a hypothesis
`h : v = e`, and `rw [l] at h` rewrites inside a hypothesis instead of the goal. `congr` proves
`f a = f b` from `a = b`.

In [34]:
theorem closeUp_surjective {n : ℕ} (σ : Perm (Fin (2 * n + 1 + 1))) (h0 : σ 0 = 0) :
    ∃ τ, closeUp τ = σ := by
  obtain ⟨⟨p, e⟩, rfl⟩ := Perm.decomposeFin.symm.surjective σ
  rw [Perm.decomposeFin_symm_apply_zero] at h0
  subst h0
  refine ⟨e.trans Fin.revPerm, ?_⟩
  unfold closeUp
  congr
  ext i
  simp

**Counting by a bijection.** Mathlib's `Fintype.card_of_bijective` says that two finite types
have the same number of elements if a bijective function maps one onto the other. Here the
function is given by name, `(f := …)`, and the bijectivity proof is the pair
`⟨injective, surjective⟩`, with both left as holes. An element `τ` of a subtype has the two
components `τ.1`, the value, and `τ.2`, the proof; `Subtype.ext` proves two elements equal from
equal values. On a proof of `A ↔ B`, `.1` and `.2` mean `.mp` and `.mpr`. `fun _ => e` ignores
its argument. `symm` turns a goal `a = b` into `b = a`. `rintro pattern` is `intro` followed by
`rcases` with that pattern.

In [35]:
/-- **Closed meanders of order `n+1` are open meanders with `2n+1` crossings.** -/
theorem closedMeanderCount_succ (n : ℕ) :
    closedMeanderCount (n + 1) = openMeanderCount (2 * n + 1) := by
  unfold closedMeanderCount openMeanderCount
  symm
  refine Fintype.card_of_bijective
    (f := fun τ => ⟨closeUp τ.1, fun _ => closeUp_zero τ.1,
      (noCross_closeUp_iff τ.1).2 τ.2⟩) ⟨?_, ?_⟩
  · intro τ τ' h
    exact Subtype.ext (closeUp_injective (congrArg Subtype.val h))
  · rintro ⟨σ, h0, hσ⟩
    obtain ⟨τ, rfl⟩ := closeUp_surjective σ (h0 (by omega))
    exact ⟨⟨τ, (noCross_closeUp_iff τ).1 hσ⟩, rfl⟩

The last proof writes $n = m + 1$, using a hypothesis `hn : 1 ≤ n` and the Mathlib lemma
`Nat.exists_eq_add_of_le'`, and finishes with `congr 1`, which reduces `f a = f b` to `a = b`,
one level deep. With the theorem proved, the table in Section 5 is no coincidence: it holds
for every $n$.

In [36]:
/-- **Closed meanders of order `n` are open meanders with `2n-1` crossings** (`n ≥ 1`). -/
theorem closedMeanderCount_eq (n : ℕ) (hn : 1 ≤ n) :
    closedMeanderCount n = openMeanderCount (2 * n - 1) := by
  obtain ⟨m, rfl⟩ := Nat.exists_eq_add_of_le' hn
  rw [closedMeanderCount_succ]
  congr 1

## 7. A search with a parity rule

`openMeanderCount` looks at all $n!$ crossing orders, which is hopeless beyond $n \approx 10$.
The library's first fast algorithm builds the crossing order one bridge at a time and abandons
a partial order as soon as it cannot be finished. A partial order is a list `l` of the bridges
visited so far. `lpt n l k` is point `k` of its path, just as `pathPt` is for a permutation:
first the south end $n+1$, then the entries of `l`, then the east end $n$. That last value is
also what `getD` returns past the end of the list.

In [37]:
/-- Boundary point `k` of a river whose crossing order is the list `l`: the south end `n+1`,
then the entries of `l`, then the east end `n` (also used past the end of `l`). -/
def lpt (n : Nat) (l : List Nat) (k : Nat) : Nat :=
  if k = 0 then n + 1 else l.getD (k - 1) n

`StepOK n q t` says that arc `t` of the partial river `q` crosses no earlier arc on its side.
When the search adds a bridge, only the new arc needs to be checked against the old ones.

In [38]:
/-- Arc `t` (joining points `t` and `t+1`) crosses no earlier arc on the same side. -/
def StepOK (n : Nat) (q : List Nat) (t : Nat) : Prop :=
  ∀ j < t, j % 2 = t % 2 →
    ¬ Interleave (lpt n q j) (lpt n q (j + 1)) (lpt n q t) (lpt n q (t + 1))

In [39]:
instance (n : Nat) (q : List Nat) (t : Nat) : Decidable (StepOK n q t) := by
  unfold StepOK; infer_instance

### The parity rule

Take an arc $j$ that is already drawn, on side $s$, with ends $a < b$. Every later arc on side
$s$ has both ends strictly inside $(a, b)$ or both outside, since it may not cross arc $j$. So
the side-$s$ arc ends still to come inside $(a, b)$ pair up, and there are an even number of
them. These ends are:

* every unvisited bridge in $(a, b)$, since each bridge eventually gets one arc on each side;
* the current point, if the next arc is on side $s$;
* the east end $n$, if the final arc $n$ is on side $s$.

If the count is odd for some drawn arc, the partial river can never be finished. At eleven
crossings this rule cuts the search from 322,080 partial rivers to 14,471.

**`Bool` and `decide`.** Besides `Prop`, Lean has `Bool`, the type of the two values `true` and
`false`. A `Bool` is data a program can compute with; a `Prop` is a statement that may have no
algorithm at all. For a decidable proposition `p`, `decide p : Bool` runs the decision
procedure. In the other direction, a `Bool` `b` becomes the proposition `b = true`. The test
"is $x$ inside arc $j$" returns a `Bool`, because the search computes with it.

In [40]:
/-- `x` lies strictly between the ends of arc `j` of the path `P`. -/
def inArc (P : Nat → Nat) (j x : Nat) : Bool :=
  decide (min (P j) (P (j + 1)) < x ∧ x < max (P j) (P (j + 1)))

`l.countP f` counts the entries `x` of the list `l` with `f x = true`. `futureEnds` adds up the
three kinds of future ends inside arc `j`, and `ParityOK` requires the count to be even for
every drawn arc.

In [41]:
/-- The number of arc ends still to come inside arc `j`, on its side of the road, when the path
`P` has drawn `t` arcs and the bridges in `r` are unvisited. -/
def futureEnds (n : Nat) (P : Nat → Nat) (t : Nat) (r : List Nat) (j : Nat) : Nat :=
  r.countP (inArc P j) + (if t % 2 = j % 2 ∧ inArc P j (P t) = true then 1 else 0) +
    (if n % 2 = j % 2 ∧ inArc P j n = true then 1 else 0)

In [42]:
/-- Every arc drawn so far encloses an even number of arc ends still to come. -/
def ParityOK (n : Nat) (q r : List Nat) : Prop :=
  ∀ j < q.length, futureEnds n (lpt n q) q.length r j % 2 = 0

In [43]:
instance (n : Nat) (q r : List Nat) : Decidable (ParityOK n q r) := by
  unfold ParityOK; infer_instance

### The depth-first search

**Definitions by pattern matching and recursion.** A function can be defined by cases on the
shape of its arguments. `def f : A → B → C` followed by lines `| pattern₁, pattern₂ => value`
gives the value for each case, tried in order. The pattern `0` matches zero, `k + 1` matches a
successor and names its predecessor `k`, a variable matches anything, and `_` matches anything
without naming it. A definition may call itself on smaller arguments, here `k` inside the case
`k + 1`. Lean checks that every case is covered and that the recursion terminates; for
*structural* recursion like this, on an argument that gets strictly smaller, the check is
automatic.

**List operations.** `l ++ l'` concatenates lists, `[x]` is the one-element list, and
`l.erase x` removes the first `x` from `l`. `(l.map f).sum` adds up the values of `f` on `l`.

`dfs n k p rem` counts the meanders whose crossing order starts with `p`, where `rem` holds the
`k` unused bridges. Each unused bridge `x` is tried next only if the new arc crosses no earlier
arc and the parity test still passes. When no bridges are left, the final arc to the east end
is checked.

In [44]:
/-- Depth-first search. `p` is the crossing order chosen so far, `rem` holds the unused bridges,
and `k = rem.length` is the recursion fuel. A bridge `x` is tried next only if the arc it
creates crosses no earlier arc and the parity test still passes. When `rem` is empty, the
final arc to the east is checked. -/
def dfs (n : Nat) : Nat → List Nat → List Nat → Nat
  | 0, p, _ => if StepOK n p p.length then 1 else 0
  | k + 1, p, rem => (rem.map fun x =>
      if StepOK n (p ++ [x]) p.length ∧ ParityOK n (p ++ [x]) (rem.erase x) then
        dfs n k (p ++ [x]) (rem.erase x)
      else 0).sum

In [45]:
#eval dfs 5 5 [] (List.range 5)

8


Eight meanders with five crossings, as `openMeanderCount 5` found by brute force.

## 8. Fast code that provably computes the same thing

`dfs` is written for proving: lists, propositions, and decision procedures assembled by type
class search. Looking up entry `k` of a list takes `k` steps. The library runs a second version
on arrays, with Boolean tests written out by hand, and proves that both compute the same
function. A compiler annotation then makes every call to `dfs` run the fast one. The speed-up
cannot change an answer, because the proof says so.

**Boolean operators.** On `Bool`, `&&` is *and*, `||` is *or*, `!b` is *not*, and `a == b` and
`a != b` test equality and inequality. Unlike `∧` and `∨`, these compute.

`allLt f t` checks `f j` for every `j < t`, by structural recursion on `t`.

In [46]:
/-- `allLt f t` checks `f j` for every `j < t`. -/
def allLt (f : Nat → Bool) : Nat → Bool
  | 0 => true
  | t + 1 => allLt f t && f t

**Induction.** `induction t with | zero => … | succ t ih => …` proves a statement for every
natural number `t`: once for `0`, and once for `t + 1` with the *induction hypothesis* `ih`,
the statement for `t`. Two more tactics appear in the proof. `constructor` splits a goal
`A ↔ B` into the two directions, or `A ∧ B` into its two halves, and `rintro` patterns like
`⟨h, ht⟩` take a conjunction apart. `Nat.lt_succ_iff_lt_or_eq` turns `j < t + 1` into
`j < t ∨ j = t`.

In [47]:
theorem allLt_iff (f : Nat → Bool) (t : Nat) : allLt f t = true ↔ ∀ j < t, f j = true := by
  induction t with
  | zero => simp [allLt]
  | succ t ih =>
    simp only [allLt, Bool.and_eq_true, ih]
    constructor
    · rintro ⟨h, ht⟩ j hj
      rcases Nat.lt_succ_iff_lt_or_eq.1 hj with h' | rfl
      · exact h j h'
      · exact ht
    · intro h
      exact ⟨fun j hj => h j (by omega), h t (by omega)⟩

**Arrays and `let`.** `Array Nat` is a sequence stored contiguously in memory, with
constant-time access `a.getD k d`, size `a.size`, and `a.push x` to append. `l.toArray` converts
a list. `let x := e` inside a definition names an intermediate value. These are the array
versions of `lpt`, `StepOK` and `ParityOK`.

In [48]:
/-- `lpt` on an array, with constant-time lookup. -/
def apt (n : Nat) (a : Array Nat) (k : Nat) : Nat :=
  if k = 0 then n + 1 else a.getD (k - 1) n

In [49]:
/-- `StepOK` as a Boolean test on an array. -/
def stepOKb (n : Nat) (a : Array Nat) (t : Nat) : Bool :=
  let c := apt n a t
  let d := apt n a (t + 1)
  allLt (fun j => j % 2 != t % 2 || !decide (Interleave (apt n a j) (apt n a (j + 1)) c d)) t

In [50]:
/-- `ParityOK` as a Boolean test on an array. -/
def parityOKb (n : Nat) (a : Array Nat) (r : List Nat) : Bool :=
  allLt (fun j => futureEnds n (apt n a) a.size r j % 2 == 0) a.size

In [51]:
def dfsArr (n : Nat) : Nat → Array Nat → List Nat → Nat
  | 0, a, _ => if stepOKb n a a.size then 1 else 0
  | k + 1, a, rem => (rem.map fun x =>
      if stepOKb n (a.push x) a.size && parityOKb n (a.push x) (rem.erase x) then
        dfsArr n k (a.push x) (rem.erase x)
      else 0).sum

In [52]:
def dfsFast (n k : Nat) (p rem : List Nat) : Nat := dfsArr n k p.toArray rem

**Three bridges between the versions.** `split` splits on the first `if` in the goal, like
`split_ifs` but without names. `simp_all` simplifies every hypothesis and the goal with each
other until nothing changes. `Bool.eq_iff_iff` turns an equation of Booleans into an `↔`, and
`decide_eq_true_iff` says `decide p = true ↔ p`. For a disjunction `h : A ∨ B`,
`h.resolve_left hna : B` uses `hna : ¬ A`. `Or.inl a : A ∨ B` and `Or.inr b : A ∨ B` build a
disjunction from either side. In a term, `show P from e` states the type `P` of `e`, and
`funext h` proves two functions equal from a proof `h` that they agree at every point.

In [53]:
theorem apt_toArray (n : Nat) (p : List Nat) (k : Nat) : apt n p.toArray k = lpt n p k := by
  unfold apt lpt
  split
  · rfl
  · simp [Array.getD, List.getD_eq_getElem?_getD]
    split <;> simp_all

In [54]:
theorem stepOKb_toArray (n : Nat) (p : List Nat) (t : Nat) :
    stepOKb n p.toArray t = decide (StepOK n p t) := by
  rw [Bool.eq_iff_iff, decide_eq_true_iff, stepOKb, allLt_iff]
  simp only [apt_toArray, Bool.or_eq_true, bne_iff_ne, ne_eq, Bool.not_eq_true',
    decide_eq_false_iff_not, StepOK]
  constructor
  · intro h j hj hpar
    exact (h j hj).resolve_left (by simpa using hpar)
  · intro h j hj
    by_cases hpar : j % 2 = t % 2
    · exact Or.inr (h j hj hpar)
    · exact Or.inl hpar

In [55]:
theorem parityOKb_toArray (n : Nat) (p r : List Nat) :
    parityOKb n p.toArray r = decide (ParityOK n p r) := by
  rw [Bool.eq_iff_iff, decide_eq_true_iff, parityOKb, allLt_iff,
    show apt n p.toArray = lpt n p from funext (apt_toArray n p), List.size_toArray]
  simp only [beq_iff_eq, ParityOK]

**Attributes, `@`, and `apply`.** `@[attr]` before a declaration attaches an *attribute*.
`@[csimp]` on a theorem `@f = @g` tells the compiler to replace `f` by `g` in all compiled code.
Lean accepts it only for a proved equation. `@f` refers to `f` with all its arguments explicit,
so `@dfs = @dfsFast` is an equation between two functions. The tactic `funext n k` reduces it
to an equation for each `n` and `k`. `apply l` proves the goal by the lemma `l` and leaves its
hypotheses as new goals. `List.map_congr_left` says that two maps over a list agree if the
functions agree on each entry.

In [56]:
@[csimp] theorem dfs_eq_dfsFast : @dfs = @dfsFast := by
  funext n k
  induction k with
  | zero => funext p rem; simp [dfs, dfsFast, dfsArr, stepOKb_toArray]
  | succ k ih =>
    funext p rem
    simp only [dfs, dfsFast, dfsArr, List.size_toArray, List.push_toArray, stepOKb_toArray,
      parityOKb_toArray]
    congr 1
    apply List.map_congr_left
    intro x _
    by_cases h : StepOK n (p ++ [x]) p.length ∧ ParityOK n (p ++ [x]) (rem.erase x) <;>
      simp [h, ih, dfsFast]

The counter is declared after the `@[csimp]` theorem, so its compiled code calls `dfsArr`.

In [57]:
/-- **Fast meander count**: the number of ways a river can cross a road `n` times.
(Declared after `dfs_eq_dfsFast`, so the compiled code uses the array version.) -/
def meanderCount (n : Nat) : Nat := dfs n n [] (List.range n)

In [58]:
#eval (List.range 14).map meanderCount

[1, 1, 1, 2, 3, 8, 14, 42, 81, 262, 538, 1828, 3926, 13820]


These fourteen values take a fraction of a second. Brute force would test
$13! \approx 6 \cdot 10^9$ permutations for the last one alone.

## 9. The search is correct

The main theorem of `Arnold/Fast.lean` is `openMeanderCount n = meanderCount n` for every `n`.
The proof has two steps.

1. The search equals filtering an explicit list `perms n (List.range n)` of all orderings, built
   by the same recursion as `dfs`, because a pruned branch contains no meanders. This needs the
   parity rule to be *sound*: a partial river that fails it really cannot be finished.
2. Permutations of `Fin n` correspond one to one with the members of that list.

**Cons and the `·` shorthand.** `x :: l` is the list `l` with `x` in front. `l.flatMap f`
applies `f`, which returns a list, to every entry and concatenates the results. `(x :: ·)` is
short for `fun l => x :: l`: a `·` in parentheses marks the argument of an anonymous function.
`perms k rem` lists all orderings of `rem`, built exactly like the search tree of `dfs`.

In [59]:
/-- All orderings of `rem` (with `k = rem.length`), generated by the same recursion as `dfs`. -/
def perms : ℕ → List ℕ → List (List ℕ)
  | 0, _ => [[]]
  | k + 1, rem => rem.flatMap fun x => (perms k (rem.erase x)).map (x :: ·)

### Bookkeeping for `NoCross`

These lemmas use only features we have seen. `NoCross P (L+1)` adds the conditions on arc `L`
to `NoCross P L`. It only looks at the points `P 0, …, P L`. And appending to a crossing order
does not move points already determined.

In [60]:
theorem noCross_succ (P : ℕ → ℕ) (L : ℕ) :
    NoCross P (L + 1) ↔ NoCross P L ∧
      ∀ j < L, j % 2 = L % 2 → ¬ Interleave (P j) (P (j + 1)) (P L) (P (L + 1)) := by
  constructor
  · intro h
    exact ⟨fun k hk => h k (by omega), h L (by omega)⟩
  · rintro ⟨h1, h2⟩ k hk
    rcases Nat.lt_succ_iff_lt_or_eq.1 hk with h | rfl
    · exact h1 k h
    · exact h2

In [61]:
theorem noCross_mono {P : ℕ → ℕ} {L L' : ℕ} (h : NoCross P L) (hL : L' ≤ L) : NoCross P L' :=
  fun k hk => h k (by omega)

In [62]:
/-- `NoCross P L` only looks at the points `P 0, …, P L`. -/
theorem noCross_congr_pts {P Q : ℕ → ℕ} {L : ℕ} (h : ∀ i ≤ L, P i = Q i) :
    NoCross P L ↔ NoCross Q L :=
  noCross_congr fun k hk j hjk _ => by
    rw [h j (by omega), h (j + 1) (by omega), h k (by omega), h (k + 1) (by omega)]

In [63]:
theorem noCross_lpt_succ (n : ℕ) (q : List ℕ) (t : ℕ) :
    NoCross (lpt n q) (t + 1) ↔ NoCross (lpt n q) t ∧ StepOK n q t :=
  noCross_succ _ _

In [64]:
/-- Appending to the crossing order does not move the points already determined. -/
theorem lpt_append (n : ℕ) (q s : List ℕ) (i : ℕ) (hi : i ≤ q.length) :
    lpt n (q ++ s) i = lpt n q i := by
  unfold lpt
  split_ifs
  · rfl
  · exact List.getD_append _ _ _ _ (by omega)

### Step 1: the search equals filtering `perms`

**Membership, permutations of lists, and `cases`.** `x ∈ l` says that `x` is an entry of `l`.
`l.Perm l'` says that `l` and `l'` have the same entries, counted with multiplicity, in any
order. `cases l with | nil => … | cons x s => …` splits on how the list `l` was built: empty,
or `x :: s`. `simp … at h` simplifies a hypothesis, as `rw … at h` rewrites one.

In [65]:
theorem mem_perms : ∀ (k : ℕ) (rem : List ℕ), rem.length = k →
    ∀ l, l ∈ perms k rem ↔ l.Perm rem := by
  intro k
  induction k with
  | zero =>
    intro rem hrem l
    rw [List.length_eq_zero_iff] at hrem
    subst hrem
    simp [perms]
  | succ k ih =>
    intro rem hrem l
    simp only [perms, List.mem_flatMap, List.mem_map]
    constructor
    · rintro ⟨x, hx, s, hs, rfl⟩
      have hlen : (rem.erase x).length = k := by rw [List.length_erase_of_mem hx]; omega
      exact List.cons_perm_iff_perm_erase.2 ⟨hx, (ih _ hlen s).1 hs⟩
    · intro h
      cases l with
      | nil => simp [List.nil_perm.1 h] at hrem
      | cons x s =>
        obtain ⟨hx, hs⟩ := List.cons_perm_iff_perm_erase.1 h
        have hlen : (rem.erase x).length = k := by rw [List.length_erase_of_mem hx]; omega
        exact ⟨x, hx, s, (ih _ hlen s).2 hs, rfl⟩

**`Nodup`, implicit lambdas, `.symm`.** `l.Nodup` says that `l` has no repeated entries.
`List.nodup_flatMap` reduces it for `flatMap` to two facts: each piece has no repeats, and
different pieces share no entries. In `fun {x y} hxy => …` the braces bind `x` and `y` as
implicit arguments of the function. For `h : a = b`, `h.symm : b = a`. `simp only [l] at h h'`
simplifies two hypotheses at once.

In [66]:
theorem nodup_perms : ∀ (k : ℕ) (rem : List ℕ), rem.Nodup → (perms k rem).Nodup := by
  intro k
  induction k with
  | zero => intro _ _; simp [perms]
  | succ k ih =>
    intro rem hrem
    refine List.nodup_flatMap.2 ⟨fun x _ => (ih _ (hrem.erase x)).map List.cons_injective, ?_⟩
    refine hrem.imp fun {x y} hxy => ?_
    intro l hl hl'
    simp only [List.mem_map] at hl hl'
    obtain ⟨_, _, rfl⟩ := hl
    obtain ⟨_, _, h⟩ := hl'
    exact hxy (List.cons.inj h).1.symm

### The parity rule is sound

`ind P j i` is `1` if point `i` of the path `P` lies strictly inside arc `j`, else `0`.

In [67]:
/-- `1` if point `i` of the path lies strictly inside arc `j`, else `0`. -/
def ind (P : ℕ → ℕ) (j i : ℕ) : ℕ := if inArc P j (P i) = true then 1 else 0

**Finite sums, `suffices`, and other induction principles.** `∑ i ∈ Finset.Ico a b, f i` is
the sum of `f i` over $a \le i < b$. `Finset` is Mathlib's type of finite sets, and
`Finset.Ico a b` is the interval $[a, b)$. `suffices H : Q by tac` states an intermediate goal
`Q`, proves the original goal from `H : Q` with `tac`, and leaves `Q` to prove next.
`induction m, hm using Nat.le_induction with | base => … | succ m hm ih => …` is induction that
starts at a number other than zero: from `hm : t + 1 ≤ m` it proves the claim for `m = t + 1`
and then from `m` to `m + 1`. `intro _` introduces a hypothesis without naming it. `if_congr`
says that two `if`s agree when their conditions and branches do, and `not_not` removes a double
negation.

The proof follows the mathematical argument: walk along the river from the current point to
the east end, and keep the count of ends inside arc `j` even, two at a time. Each later
same-side arc puts both of its ends inside arc `j` or neither.

In [68]:
/-- **The parity argument.** Fix a drawn arc `j < t` of a river `P` without self-crossings.
The later arcs on the same side (indices `k ≥ t` with `k ≡ j`) each have both ends inside arc
`j` or both outside, so they put an even number of ends inside it. Going along the river, the
points `t+1, …, n` are each the end of exactly one such arc. The current point `t` and the east
end `n+1` are ends only when their arc is on that side. -/
theorem parity_future (P : ℕ → ℕ) (n t j : ℕ) (hnc : NoCross P (n + 1)) (hj : j < t)
    (ht : t ≤ n) :
    (∑ i ∈ Finset.Ico (t + 1) (n + 1), ind P j i + (if t % 2 = j % 2 then ind P j t else 0) +
      (if n % 2 = j % 2 then ind P j (n + 1) else 0)) % 2 = 0 := by
  -- A later same-side arc `k` has both ends inside arc `j` or neither.
  have harc : ∀ k, j < k → k ≤ n → k % 2 = j % 2 → ind P j (k + 1) = ind P j k := by
    intro k hjk hkn hpar
    have h := hnc k (by omega) j hjk hpar.symm
    unfold Interleave at h
    rw [not_not] at h
    unfold ind inArc
    simp only [decide_eq_true_eq]
    exact if_congr h.symm rfl rfl
  have hind : ∀ i, ind P j i ≤ 1 := fun i => by unfold ind; split <;> omega
  suffices H : ∀ m, t + 1 ≤ m → m ≤ n + 1 →
      (∑ i ∈ Finset.Ico (t + 1) m, ind P j i + (if t % 2 = j % 2 then ind P j t else 0) +
        (if (m - 1) % 2 = j % 2 then ind P j m else 0)) % 2 = 0 by
    simpa using H (n + 1) (by omega) le_rfl
  intro m hm
  induction m, hm using Nat.le_induction with
  | base =>
    intro _
    simp only [Finset.Ico_self, Finset.sum_empty, zero_add, Nat.add_sub_cancel]
    by_cases hp : t % 2 = j % 2
    · simp only [hp, ↓reduceIte, harc t hj ht hp]
      omega
    · simp [hp]
  | succ m hm ih =>
    intro hm'
    have h := ih (by omega)
    rw [Finset.sum_Ico_succ_top (by omega), Nat.add_sub_cancel]
    by_cases hp : m % 2 = j % 2
    · simp only [show ¬ (m - 1) % 2 = j % 2 by omega, ↓reduceIte, add_zero] at h
      simp only [hp, ↓reduceIte, harc m (by omega) (by omega) hp]
      omega
    · simp only [show (m - 1) % 2 = j % 2 by omega, ↓reduceIte] at h
      simp only [hp, ↓reduceIte, add_zero]
      omega

**A recursive proof.** A theorem can be proved by pattern matching and recursion, exactly like a
function is defined; the recursive call is the induction hypothesis. `countP_eq_sum` relates
counting the entries of `s` to summing over their positions on the path `q ++ s`, by recursion
on `s`.

In [69]:
/-- Counting the entries of `s` with a property, by their positions on the path `q ++ s`. -/
theorem countP_eq_sum (n : ℕ) (g : ℕ → Bool) : ∀ (s q : List ℕ),
    s.countP g = ∑ i ∈ Finset.Ico (q.length + 1) (q.length + s.length + 1),
      (if g (lpt n (q ++ s) i) = true then 1 else 0)
  | [], q => by simp
  | x :: s, q => by
    rw [List.countP_cons, countP_eq_sum n g s (q ++ [x]),
      Finset.sum_eq_sum_Ico_succ_bot (show q.length + 1 < q.length + (x :: s).length + 1 by simp)]
    have hx : lpt n (q ++ x :: s) (q.length + 1) = x := by
      simp [lpt, List.getD_eq_getElem?_getD]
    simp only [List.length_append, List.length_cons, hx, List.append_assoc,
      List.singleton_append]
    rw [show q.length + (s.length + 1) + 1 = q.length + 1 + s.length + 1 by omega]
    by_cases hg : g x = true <;> simp [hg]
    omega

**Rewriting backwards.** `rw [← h]` rewrites with `h : a = b` from right to left, replacing `b`
by `a`. With that, soundness of the parity rule follows from `parity_future`: if `q ++ s` is a
meander and `s` uses the bridges `r`, then `q` passes the parity test with remaining bridges
`r`.

In [70]:
/-- **The parity test is sound**: if `q ++ s` is a meander and `s` uses the bridges `r`, then
`q` passes the parity test with remaining bridges `r`. -/
theorem parityOK_of_noCross (n : ℕ) (q s r : List ℕ) (hlen : q.length + s.length = n)
    (hperm : s.Perm r) (hnc : NoCross (lpt n (q ++ s)) (n + 1)) : ParityOK n q r := by
  intro j hj
  have hPq : ∀ i ≤ q.length, lpt n q i = lpt n (q ++ s) i := fun i hi =>
    (lpt_append n q s i hi).symm
  have hin : inArc (lpt n q) j = inArc (lpt n (q ++ s)) j := by
    funext x
    simp only [inArc, hPq j (by omega), hPq (j + 1) (by omega)]
  have hE : lpt n (q ++ s) (n + 1) = n := by
    simp [lpt, List.getD_eq_getElem?_getD, hlen]
  have h := parity_future (lpt n (q ++ s)) n q.length j hnc hj (by omega)
  unfold futureEnds
  rw [hin, ← hperm.countP_eq, countP_eq_sum n _ s q, hPq q.length le_rfl, hlen]
  have e2 : (if q.length % 2 = j % 2 ∧
      inArc (lpt n (q ++ s)) j (lpt n (q ++ s) q.length) = true then 1 else 0) =
      (if q.length % 2 = j % 2 then ind (lpt n (q ++ s)) j q.length else 0) := by
    unfold ind; split_ifs <;> simp_all
  have e3 : (if n % 2 = j % 2 ∧ inArc (lpt n (q ++ s)) j n = true then 1 else 0) =
      (if n % 2 = j % 2 then ind (lpt n (q ++ s)) j (n + 1) else 0) := by
    unfold ind; rw [hE]; split_ifs <;> simp_all
  rw [e2, e3]
  exact h

**`obtain` with a type, and `swap`.** `obtain rfl : n = p.length := by omega` proves the
equation and substitutes it at once. `swap` exchanges the first two goals, so the second can be
proved first. With these, the search equals the count of meanders among the orderings in
`perms`, at every node of the search tree.

In [71]:
theorem dfs_eq (n : ℕ) : ∀ (k : ℕ) (p rem : List ℕ), rem.length = k → p.length + k = n →
    NoCross (lpt n p) p.length →
    dfs n k p rem = (perms k rem).countP (fun s => decide (NoCross (lpt n (p ++ s)) (n + 1))) := by
  intro k
  induction k with
  | zero =>
    intro p rem _ hp hpre
    obtain rfl : n = p.length := by omega
    simp only [dfs, perms, List.countP_cons, List.countP_nil, List.append_nil, zero_add]
    by_cases h : StepOK p.length p p.length <;> simp [noCross_lpt_succ, h, hpre]
  | succ k ih =>
    intro p rem hrem hp hpre
    simp only [dfs, perms, List.countP_flatMap]
    congr 1
    apply List.map_congr_left
    intro x hx
    simp only [Function.comp_apply, List.countP_map]
    have hlen : (rem.erase x).length = k := by
      rw [List.length_erase_of_mem hx]; omega
    split_ifs with hstep
    · rw [ih (p ++ [x]) (rem.erase x) hlen (by simp; omega)]
      · simp [Function.comp_def, List.append_assoc]
      · rw [List.length_append, List.length_singleton, noCross_lpt_succ]
        refine ⟨?_, hstep.1⟩
        exact (noCross_congr_pts fun i hi => lpt_append n p [x] i hi).2 hpre
    · symm
      rw [List.countP_eq_zero]
      intro s hsmem hs
      simp only [Function.comp_apply, decide_eq_true_eq] at hs
      apply hstep
      refine ⟨?_, ?_⟩
      swap
      · -- The parity test is sound: a finished river passes it at every stage.
        have hsperm := (mem_perms k (rem.erase x) hlen s).1 hsmem
        have hslen : s.length = k := by rw [hsperm.length_eq, hlen]
        exact parityOK_of_noCross n (p ++ [x]) s (rem.erase x) (by simp; omega) hsperm
          (by simpa [List.append_assoc] using hs)
      have h1 : NoCross (lpt n ((p ++ [x]) ++ s)) (p.length + 1) :=
        noCross_mono (by simpa [List.append_assoc] using hs) (by omega)
      have h2 : NoCross (lpt n (p ++ [x])) (p.length + 1) :=
        (noCross_congr_pts fun i hi => lpt_append n (p ++ [x]) s i (by simp; omega)).1 h1
      exact ((noCross_lpt_succ n _ _).1 h2).2

### Step 2: permutations of `Fin n` are the orderings in `perms`

**Lists from functions.** `List.ofFn f`, for `f : Fin n → α`, is the list `[f 0, …, f (n-1)]`.
`toList σ` is the crossing order of `σ` as a list of numbers.

In [72]:
/-- The crossing order of `σ` as a list of natural numbers. -/
def toList {n : ℕ} (σ : Perm (Fin n)) : List ℕ := List.ofFn fun i => (σ i : ℕ)

In [73]:
theorem pathPt_eq_lpt {n : ℕ} (σ : Perm (Fin n)) (k : ℕ) : pathPt σ k = lpt n (toList σ) k := by
  unfold pathPt lpt toList
  by_cases h0 : k = 0
  · simp [h0]
  · simp only [h0, ↓reduceIte, List.getD_eq_getElem?_getD, List.getElem?_ofFn]
    split_ifs <;> rfl

In [74]:
theorem toList_perm {n : ℕ} (σ : Perm (Fin n)) : (toList σ).Perm (List.range n) := by
  refine (List.perm_ext_iff_of_nodup (List.nodup_ofFn.2 fun i j h => σ.injective (Fin.ext h))
    List.nodup_range).2 fun a => ?_
  simp only [List.mem_ofFn, List.mem_range]
  constructor
  · rintro ⟨i, rfl⟩
    exact (σ i).isLt
  · intro ha
    exact ⟨σ.symm ⟨a, ha⟩, by simp⟩

`congrFun h x : f x = g x` applies an equation of functions `h : f = g` at a point.

In [75]:
theorem toList_injective {n : ℕ} : Function.Injective (toList (n := n)) := fun _ _ h =>
  Equiv.ext fun i => Fin.ext (congrFun (List.ofFn_injective h) i)

**Indexing with a proof.** `l[i]'h` is entry `i` of the list `l`, where `h : i < l.length`
proves that the index is in range, so no default is needed. Below, `let f : Fin n → Fin n :=
…` defines inside a proof the function that a list `l` describes. `Equiv.ofBijective` turns a
bijective function into an equivalence, and on a finite type an injective function is
bijective (`Finite.injective_iff_bijective`).

In [76]:
theorem exists_toList_eq {n : ℕ} {l : List ℕ} (h : l.Perm (List.range n)) :
    ∃ σ : Perm (Fin n), toList σ = l := by
  have hlen : l.length = n := by simpa using h.length_eq
  have hnd : l.Nodup := h.nodup_iff.2 List.nodup_range
  have hlt : ∀ x ∈ l, x < n := fun x hx => List.mem_range.1 (h.subset hx)
  let f : Fin n → Fin n := fun i => ⟨l[i.1]'(by omega), hlt _ (List.getElem_mem _)⟩
  have hf : Function.Injective f := fun i j hij =>
    Fin.ext ((hnd.getElem_inj_iff).1 (congrArg Fin.val hij))
  refine ⟨Equiv.ofBijective f (Finite.injective_iff_bijective.1 hf), ?_⟩
  apply List.ext_getElem (by simp [toList, hlen])
  intro i _ _
  simp [toList, f]

**Instance arguments.** In `(Q : List ℕ → Prop) [DecidablePred Q]`, the square brackets take
an *instance* argument: callers never pass it, because Lean finds it by type class search.
`DecidablePred Q` says that `Q l` is decidable for every `l`. `Finset.card_bij` proves that two
finite sets have the same size, given a map between them that sends one into the other, is
injective, and hits everything.

In [77]:
theorem card_eq_countP (n : ℕ) (Q : List ℕ → Prop) [DecidablePred Q] :
    Fintype.card {σ : Perm (Fin n) // Q (toList σ)} =
      (perms n (List.range n)).countP (fun l => decide (Q l)) := by
  have hmem := mem_perms n (List.range n) (by simp)
  rw [Fintype.card_subtype, List.countP_eq_length_filter,
    ← List.toFinset_card_of_nodup ((nodup_perms _ _ List.nodup_range).filter _)]
  apply Finset.card_bij (fun σ _ => toList σ)
  · intro σ hσ
    simp only [Finset.mem_filter, Finset.mem_univ, true_and] at hσ
    simpa [hmem] using ⟨toList_perm σ, hσ⟩
  · intro σ _ τ _ h
    exact toList_injective h
  · intro l hl
    simp only [List.mem_toFinset, List.mem_filter, decide_eq_true_eq, hmem] at hl
    obtain ⟨σ, rfl⟩ := exists_toList_eq hl.1
    exact ⟨σ, by simpa using hl.2, rfl⟩

### The main theorems

`Fintype.card_congr` says that equivalent types have the same number of elements, and
`Equiv.subtypeEquivRight` says that two subtypes of the same type are equivalent when their
conditions are equivalent. From `h : P` and `hn : ¬ P`, `absurd h hn` proves anything; the
first theorem uses it to dismiss the impossible `k < 0`.

In [78]:
/-- **The fast search counts exactly Arnold's meanders**, for every `n`. -/
theorem openMeanderCount_eq_meanderCount (n : ℕ) : openMeanderCount n = meanderCount n := by
  have h : openMeanderCount n =
      Fintype.card {σ : Perm (Fin n) // NoCross (lpt n (toList σ)) (n + 1)} :=
    Fintype.card_congr (Equiv.subtypeEquivRight fun σ => by
      rw [IsMeander, show pathPt σ = lpt n (toList σ) from funext (pathPt_eq_lpt σ)])
  rw [h, card_eq_countP n (fun l => NoCross (lpt n l) (n + 1)), meanderCount,
    dfs_eq n n [] (List.range n) (by simp) (by simp) (fun k hk => absurd hk (Nat.not_lt_zero k))]
  simp

In [79]:
/-- Closed meanders of order `n ≥ 1`, counted by the fast search. -/
theorem closedMeanderCount_eq_meanderCount (n : ℕ) (hn : 1 ≤ n) :
    closedMeanderCount n = meanderCount (2 * n - 1) := by
  rw [closedMeanderCount_eq n hn, openMeanderCount_eq_meanderCount]

**`#print axioms`.** Lean proofs may use a few standard axioms of mathematics beyond its type
theory: `propext` (equivalent propositions are equal), `Quot.sound` (for quotient types), and
`Classical.choice` (the axiom of choice). `#print axioms t` lists the axioms that the theorem `t`
depends on, through all the lemmas it uses. Nothing else is assumed.

In [80]:
#print axioms openMeanderCount_eq_meanderCount

'Arnold.openMeanderCount_eq_meanderCount' depends on axioms: [propext, Classical.choice, Quot.sound]


Now a value of Arnold's number can be *proved* with the fast search: rewrite with the theorem,
then let the kernel run `meanderCount`. The `;` runs the two tactics in turn. (Brute force
would check $8! = 40{,}320$ permutations inside the kernel; the search visits about 500
partial rivers.)

In [81]:
example : openMeanderCount 8 = 81 := by rw [openMeanderCount_eq_meanderCount]; decide

## 10. The transfer matrix

The search follows the river. The transfer matrix scans the road instead, from west to east:
the bridges $0, \ldots, m-1$, then the east end $E = m$, which has one arc on side $m \bmod 2$.
The south end $S = m+1$, with its one arc below, comes last and needs no step. At each point the
machine chooses, for each side, whether the point *opens* a new arc or *closes* an open one. Arcs
on one side never cross, so the open arcs on each side form a stack, and a point always closes
the top one.

The river west of the cut falls into pieces, each joining two open arcs (or one open arc and
$E$). The *state* records, for each open arc, which arc its piece leads to. Joining the two ends
of one piece would close a loop, so the machine refuses that move. A whole river is accepted
when, after $E$, exactly one arc is open: below the road, with its piece leading to $E$, ready
for $S$ to close it. Equal states are merged and their counts added, which is what makes the
method fast.

**`end` and nested namespaces.** The machine lives in `Arnold.TM`. We close `Arnold` and open
`Arnold.TM`, a namespace inside it, so names from `Arnold` stay available without the prefix.

**Inductive types.** `inductive T where | c₁ … | c₂ …` defines a new type whose values are
built by the listed *constructors*, and only by them. A reference to an open arc is either the
east end `E`, or `s σ i`, position `i` in the stack of side `σ` (`true` above the road). `Ref.E`
and `Ref.s true 3` are values of the type `Ref`.

**`deriving`.** `deriving DecidableEq, Repr, Inhabited` asks Lean to write three instances: an
algorithm deciding whether two `Ref`s are equal, a way to print them (used by `#eval`), and a
default value.

In [82]:
end Arnold

namespace Arnold.TM

/-- A reference to an open arc: the east end `E`, or position `i` (counting from the bottom)
in the stack of side `σ` (`true` = above the road, `false` = below). -/
inductive Ref where
  | E
  | s (σ : Bool) (i : Nat)
  deriving DecidableEq, Repr, Inhabited

**Structures.** `structure St where` followed by fields defines a record type, here a state
with two stacks of references. `s.up` and `s.dn` are its fields, and `⟨u, d⟩` builds one.

In [83]:
/-- The state at a cut: for each open arc, in stack order, where its piece of river leads. -/
structure St where
  up : List Ref
  dn : List Ref
  deriving DecidableEq, Repr, Inhabited

**Functions in a type's namespace.** The stack operations are declared in the namespace `St`,
so dot notation finds them: `s.push σ r` means `St.push s σ r`. `{ s with up := l }` is `s`
with the field `up` replaced. `l.dropLast` removes the last entry, and `l.set i r` replaces
entry `i`. In patterns, `.E` and `.s σ i` abbreviate `Ref.E` and `Ref.s σ i`, since Lean knows
the type.

In [84]:
namespace St

def stk (s : St) (σ : Bool) : List Ref := if σ then s.up else s.dn
def setStk (s : St) (σ : Bool) (l : List Ref) : St :=
  if σ then { s with up := l } else { s with dn := l }
def push (s : St) (σ : Bool) (r : Ref) : St := s.setStk σ (s.stk σ ++ [r])
def pop (s : St) (σ : Bool) : St := s.setStk σ (s.stk σ).dropLast
/-- Make the open arc `p` lead to `r`. -/
def setRef (s : St) : Ref → Ref → St
  | .E, _ => s
  | .s σ i, r => s.setStk σ ((s.stk σ).set i r)

end St

**`Option` and `match`.** A step can fail, and `Option α` holds either `none` (no value) or
`some a` for `a : α`. `l.getLast? : Option α` is the last entry of `l`, if any. `match e with |
pattern => value …` chooses a case by the shape of `e`, as a definition by pattern matching does.
`openClose s σ` opens an arc on side `σ` and closes the top arc on the other side `!σ`. The new
arc continues the closed arc's piece.

In [85]:
/-- Open an arc on side `σ` and close the top arc on side `!σ`: the new arc continues the closed
arc's piece. -/
def openClose (s : St) (σ : Bool) : Option St :=
  match (s.stk (!σ)).getLast? with
  | none => none
  | some p => some (((s.pop (!σ)).setRef p (.s σ (s.stk σ).length)).push σ p)

**Matching several values.** `match a with | (true, true) => …` matches a pair against literal
patterns, and `match e₁, e₂ with | some p, some q => … | _, _ => …` matches two values at
once. `step m x s a` processes point `x` with action `a = (above, below)`, where `true` means
*open*. A bridge has four actions. The east end `x = m` uses only its side and reads `a.1`, the
first component of the pair. When both tops close, the machine refuses if the upper top's piece
leads to the lower top: that would close a loop.

In [86]:
/-- Process point `x` with action `a = (above, below)` (`true` = open). Point `x = m` is the
east end, which uses only its side `m % 2` and reads `a.1`. -/
def step (m x : Nat) (s : St) (a : Bool × Bool) : Option St :=
  if x < m then
    match a with
    | (true, true) =>
      some ((s.push true (.s false (s.stk false).length)).push false
        (.s true (s.stk true).length))
    | (true, false) => openClose s true
    | (false, true) => openClose s false
    | (false, false) =>
      match (s.stk true).getLast?, (s.stk false).getLast? with
      | some pu, some pl =>
        if pu = .s false ((s.stk false).length - 1) then none
        else some ((((s.pop true).pop false).setRef pu pl).setRef pl pu)
      | _, _ => none
  else
    let e := m % 2 == 1
    if a.1 then some (s.push e .E)
    else
      match (s.stk e).getLast? with
      | none => none
      | some p => if p = .E then none else some ((s.pop e).setRef p .E)

In [87]:
/-- The actions available at point `x`: both sides free at a bridge, one side at `E`. -/
def acts (m x : Nat) : List (Bool × Bool) :=
  if x < m then [(true, true), (true, false), (false, true), (false, false)]
  else [(true, true), (false, false)]
def init : St := ⟨[], []⟩

In [88]:
/-- After `E`: one arc open, below the road, leading to `E`. Then `S` closes it. -/
def final : St := ⟨[], [.E]⟩

One step from the empty state at the first of three bridges, opening above and below. `Repr`
lets `#eval` print the state: one arc above, whose piece leads to the arc below, and vice versa.

In [89]:
#eval step 3 0 init (true, true)

some { up := [Arnold.TM.Ref.s false 0], dn := [Arnold.TM.Ref.s true 0] }


### Counting with merged states

A *layer* is a list of pairs (state, number of action sequences that reach it). Merging sorts
the layer by a numeric key and adds up neighbours with equal states. Any key is correct; a good
one puts equal states next to each other.

**Bool from Prop, again.** In `a < b || (a == b && lexLe l l')`, the proposition `a < b` stands
where a `Bool` is expected, and Lean inserts `decide` automatically.

In [90]:
def refCode : Ref → Nat
  | .E => 0
  | .s false i => 2 * i + 1
  | .s true i => 2 * i + 2

In [91]:
/-- A sort key for states. Any key works for correctness; a good one makes equal states adjacent. -/
def key (s : St) : List Nat := s.up.length :: (s.up.map refCode ++ s.dn.map refCode)

In [92]:
def lexLe : List Nat → List Nat → Bool
  | [], _ => true
  | _ :: _, [] => false
  | a :: l, b :: l' => a < b || (a == b && lexLe l l')

**Well-founded recursion.** `mergeAdj` calls itself on `(k, s, c + c') :: rest`, which is not
a piece of its argument, so the recursion is not structural. Lean then tries to prove that some
measure decreases, here the length of the list, and accepts the definition when it succeeds.
Triples are nested pairs: `(k, s, c)` is `(k, (s, c))`, so for a triple `t`, `t.2.1` is `s`.

In [93]:
/-- Add up the counts of adjacent equal states. -/
def mergeAdj : List (List Nat × St × Nat) → List (St × Nat)
  | [] => []
  | [(_, s, c)] => [(s, c)]
  | (k, s, c) :: (k', s', c') :: rest =>
    if s = s' then mergeAdj ((k, s, c + c') :: rest) else (s, c) :: mergeAdj ((k', s', c') :: rest)

`l.mergeSort le` sorts a list by the Boolean comparison `le`.

In [94]:
/-- Sort by key and merge equal neighbours. -/
def compress (L : List (St × Nat)) : List (St × Nat) :=
  mergeAdj ((L.map fun sc => (key sc.1, sc.1, sc.2)).mergeSort fun a b => lexLe a.1 b.1)

**`filterMap` and `Option.map`.** `l.filterMap f` applies `f : α → Option β` to every entry and
keeps the values inside the `some`s. `o.map f` applies `f` inside an `Option`. `expand` makes
every successor of every state of a layer, carrying the counts along.

In [95]:
/-- All successors of a layer at point `x`, keeping multiplicities. -/
def expand (m x : Nat) (L : List (St × Nat)) : List (St × Nat) :=
  L.flatMap fun sc => (acts m x).filterMap fun a => (step m x sc.1 a).map fun s' => (s', sc.2)

A state with more open arcs on a side than the remaining points can close is dropped at once.
`cap m k σ` counts the points $k, \ldots, m$ that have an arc on side `σ`. `l.filter f` keeps the
entries with `f x = true`.

In [96]:
/-- How many of the points `k, …, m` still to come have an arc on side `σ`: each can close at
most one open arc there. -/
def cap (m k : Nat) (σ : Bool) : Nat :=
  if k ≤ m then (m - k) + (if (m % 2 == 1) == σ then 1 else 0) else 0

In [97]:
/-- A state can still be finished only if the remaining points can close its open arcs (all of
them above the road, all but the one `S` closes below). -/
def viable (m k : Nat) (s : St) : Bool :=
  (s.stk true).length ≤ cap m k true && (s.stk false).length ≤ cap m k false + 1

**Functions as parameters.** `layerWith` takes the merge step `comp` as an argument, a function
from layers to layers. Correctness is proved once, for any merge that keeps weighted sums, and
then applies to both merges the library uses.

In [98]:
/-- The layer after processing points `0, …, k-1`, merging equal states with `comp` and keeping
only viable states. -/
def layerWith (comp : List (St × Nat) → List (St × Nat)) (m : Nat) : Nat → List (St × Nat)
  | 0 => [(init, 1)]
  | k + 1 => (comp (expand m k (layerWith comp m k))).filter fun sc => viable m (k + 1) sc.1

In [99]:
def tmCountWith (comp : List (St × Nat) → List (St × Nat)) (m : Nat) : Nat :=
  ((layerWith comp m (m + 1)).map fun sc => if sc.1 = final then sc.2 else 0).sum

In [100]:
/-- **Transfer-matrix meander count** (compiled: merges by sorting). -/
def tmCount (m : Nat) : Nat := tmCountWith compress m

The second merge inserts states one at a time by structural recursion, which Lean's kernel can
evaluate efficiently. `l.foldr f b` combines the entries of `l` from the right:
`f x₁ (f x₂ (… (f xₖ b)))`.

In [101]:
/-- Add one state to a list of distinct states. Structural recursion, so the kernel can run it. -/
def insertAdd (sc : St × Nat) : List (St × Nat) → List (St × Nat)
  | [] => [sc]
  | (s, c) :: L => if s = sc.1 then (s, c + sc.2) :: L else (s, c) :: insertAdd sc L
def compressK (L : List (St × Nat)) : List (St × Nat) := L.foldr insertAdd []

In [102]:
/-- The same count with a kernel-friendly merge, for values checked by `decide`. -/
def tmCountK (m : Nat) : Nat := tmCountWith compressK m

In [103]:
#eval (List.range 25).map tmCount

[1, 1, 1, 2, 3, 8, 14, 42, 81, 262, 538, 1828, 3926, 13820, 30694, 110954, 252939, 933458, 2172830, 8152860, 19304190,
  73424650, 176343390, 678390116, 1649008456]


These are the values up to $n = 24$, in about a second, even interpreted. The compiled program
`meanders` reaches $n = 32$ in a minute and a half.

## 11. Merging states is harmless

`Arnold/TM/Count.lean` proves that `tmCount m` equals the number of action sequences, one action
per point $0, \ldots, m$, that drive the machine from `init` to `final`. The key idea: the number
of accepted continuations from a state depends only on the state. So adding up the counts of
equal states, or dropping states with no accepted continuation, changes no total.

**`set_option`.** `set_option name value` changes a setting of Lean for the rest of the file
(or, followed by `in`, for one declaration). The file switches off warnings about deprecated
library names.

**`Option.bind`.** `o.bind f` is `none` if `o` is `none`, and `f a` if `o` is `some a`. It chains
steps that may fail: `runFrom` runs the machine on a whole action sequence.

In [104]:
set_option linter.deprecated false

/-- All action sequences for `f` consecutive points starting at `x`. -/
def sufWords (m : ℕ) : ℕ → ℕ → List (List (Bool × Bool))
  | 0, _ => [[]]
  | f + 1, x => (acts m x).flatMap fun a => (sufWords m f (x + 1)).map (a :: ·)

In [105]:
/-- Run the machine from state `s` at point `x` on an action sequence. -/
def runFrom (m : ℕ) : ℕ → St → List (Bool × Bool) → Option St
  | _, s, [] => some s
  | x, s, a :: w => (step m x s a).bind fun s' => runFrom m (x + 1) s' w

In [106]:
/-- The number of accepted continuations of length `f` from state `s` at point `x`. -/
def cntFrom (m f x : ℕ) (s : St) : ℕ :=
  (sufWords m f x).countP fun w => runFrom m x s w = some final

In [107]:
theorem cntFrom_zero (m x : ℕ) (s : St) : cntFrom m 0 x s = if s = final then 1 else 0 := by
  by_cases h : s = final <;> simp [cntFrom, sufWords, runFrom, h]

**`cases` with an equation.** `cases h : e with | none => … | some s' => …` splits on the
value of `e` and records which one it is as `h : e = none` or `h : e = some s'`.

In [108]:
theorem cntFrom_succ (m f x : ℕ) (s : St) :
    cntFrom m (f + 1) x s = ((acts m x).map fun a =>
      match step m x s a with
      | none => 0
      | some s' => cntFrom m f (x + 1) s').sum := by
  unfold cntFrom
  rw [sufWords, List.countP_flatMap]
  congr 1
  apply List.map_congr_left
  intro a _
  simp only [Function.comp_apply, List.countP_map]
  cases h : step m x s a with
  | none => simp [Function.comp_def, runFrom, h]
  | some s' => simp [Function.comp_def, runFrom, h]

### Merging preserves weighted sums

`wsum g L` is $\sum c \cdot g(s)$ over the pairs $(s, c)$ of the layer `L`.

In [109]:
/-- The weighted sum `Σ c * g s` of a layer. -/
def wsum (g : St → ℕ) (L : List (St × ℕ)) : ℕ := (L.map fun sc => sc.2 * g sc.1).sum

**Functional induction and `ring`.** For a function defined by recursion, Lean generates an
induction principle that follows its cases: `induction T using mergeAdj.induct with | case1 =>
… | case2 … | case3 … | case4 …` has one case for each branch of `mergeAdj`, with an induction
hypothesis wherever `mergeAdj` calls itself. `ring` proves equations that hold in every
commutative ring, such as $(c + c') g = c g + c' g$. A location `at h ⊢` makes `simp` or `rw`
work on the hypothesis `h` and on the goal, written `⊢`.

In [110]:
theorem mergeAdj_wsum (g : St → ℕ) (T : List (List ℕ × St × ℕ)) :
    wsum g (mergeAdj T) = (T.map fun t => t.2.2 * g t.2.1).sum := by
  induction T using mergeAdj.induct with
  | case1 => simp [mergeAdj, wsum]
  | case2 k s c => simp [mergeAdj, wsum]
  | case3 k c k' s' c' rest ih =>
    simp only [mergeAdj, ↓reduceIte, ih, List.map_cons, List.sum_cons]
    ring
  | case4 k s c k' s' c' rest hne ih =>
    simp only [mergeAdj, hne, ↓reduceIte, wsum, List.map_cons, List.sum_cons] at ih ⊢
    rw [ih]

`List.mergeSort_perm` says sorting permutes the list, and a sum over a permuted list is the
same. In the next proofs, `rw [← wsum]` rewrites with the definition of `wsum` backwards,
folding its body back into the name.

In [111]:
theorem compress_wsum (g : St → ℕ) (L : List (St × ℕ)) : wsum g (compress L) = wsum g L := by
  unfold compress
  rw [mergeAdj_wsum, ((List.mergeSort_perm _ _).map _).sum_eq]
  simp [wsum, Function.comp_def]

In [112]:
theorem filterMap_wsum (m x c : ℕ) (s : St) (g : St → ℕ) (as : List (Bool × Bool)) :
    wsum g (as.filterMap fun a => (step m x s a).map fun s' => (s', c)) =
      c * (as.map fun a => match step m x s a with
        | none => 0
        | some s' => g s').sum := by
  induction as with
  | nil => simp [wsum]
  | cons a as ih =>
    cases h : step m x s a with
    | none => simp [h, ih]
    | some s' =>
      simp only [wsum, List.filterMap_cons, h, Option.map_some, List.map_cons, List.sum_cons,
        List.map, List.sum_cons] at ih ⊢
      rw [ih]
      ring

In [113]:
theorem expand_wsum (m x f : ℕ) (L : List (St × ℕ)) :
    wsum (cntFrom m f (x + 1)) (expand m x L) = wsum (cntFrom m (f + 1) x) L := by
  induction L with
  | nil => simp [expand, wsum]
  | cons sc L ih =>
    have h1 : expand m x (sc :: L) = ((acts m x).filterMap fun a =>
        (step m x sc.1 a).map fun s' => (s', sc.2)) ++ expand m x L := by
      simp [expand]
    rw [h1]
    simp only [wsum, List.map_append, List.sum_append] at ih ⊢
    rw [ih, ← wsum, filterMap_wsum]
    simp [cntFrom_succ]

### Dropping states that cannot be finished

These lemmas track the stack heights through a step. `cases σ <;> cases τ` tries all four
combinations of two Booleans.

In [114]:
theorem length_setRef (s : St) (p r : Ref) (σ : Bool) :
    ((s.setRef p r).stk σ).length = (s.stk σ).length := by
  cases p with
  | E => rfl
  | s τ i => cases τ <;> cases σ <;> simp [St.setRef, St.setStk, St.stk]

In [115]:
theorem stk_push_len (s : St) (σ τ : Bool) (r : Ref) :
    ((s.push σ r).stk τ).length = (s.stk τ).length + (if σ = τ then 1 else 0) := by
  cases σ <;> cases τ <;> simp [St.push, St.setStk, St.stk]

In [116]:
theorem stk_pop_len (s : St) (σ τ : Bool) :
    ((s.pop σ).stk τ).length = (s.stk τ).length - (if σ = τ then 1 else 0) := by
  cases σ <;> cases τ <;> simp [St.pop, St.setStk, St.stk]

In [117]:
theorem len_pos_of_getLast {l : List Ref} {p : Ref} (h : l.getLast? = some p) : 0 < l.length := by
  cases l with
  | nil => simp at h
  | cons _ _ => simp

**Splitting a step.** `rcases a with ⟨_ | _, _ | _⟩` splits the action pair into its four
Boolean cases (`false` is the first constructor of `Bool`, `true` the second). `split_ifs at h`
splits on the `if`s in a hypothesis. `dsimp only at h` unfolds definitions in `h` without using
lemmas. `reduceCtorEq` is a simproc that turns an equation between different constructors, such
as `none = some s`, into `False`, and `Option.some.injEq` turns `some a = some b` into `a = b`.
`if_pos hx` rewrites `if c then a else b` to `a` given `hx : c`, and `if_neg` to `b`.

In [118]:
/-- A step closes at most one open arc on each side, and only on the sides the point has. -/
theorem step_len {m x : ℕ} {s s' : St} {a : Bool × Bool} (h : step m x s a = some s')
    (σ : Bool) : (s.stk σ).length ≤
      (s'.stk σ).length + (if x < m ∨ (m % 2 == 1) = σ then 1 else 0) := by
  unfold step at h
  by_cases hx : x < m
  · rw [if_pos hx] at h
    simp only [hx, true_or, ↓reduceIte]
    rcases a with ⟨_ | _, _ | _⟩
    · simp only at h
      cases hu : (s.stk true).getLast? <;> cases hl : (s.stk false).getLast? <;>
        rw [hu, hl] at h <;> simp only [reduceCtorEq] at h
      split_ifs at h
      simp only [Option.some.injEq] at h
      subst h
      rw [length_setRef, length_setRef, stk_pop_len, stk_pop_len]
      cases σ <;> simp <;> omega
    · unfold openClose at h
      cases ht : (s.stk (!false)).getLast? <;> rw [ht] at h <;> simp only [reduceCtorEq,
        Option.some.injEq] at h
      subst h
      rw [stk_push_len, length_setRef, stk_pop_len]
      cases σ <;> simp <;> omega
    · unfold openClose at h
      cases ht : (s.stk (!true)).getLast? <;> rw [ht] at h <;> simp only [reduceCtorEq,
        Option.some.injEq] at h
      subst h
      rw [stk_push_len, length_setRef, stk_pop_len]
      cases σ <;> simp <;> omega
    · simp only [Option.some.injEq] at h
      subst h
      rw [stk_push_len, stk_push_len]
      omega
  · rw [if_neg hx] at h
    simp only [hx, false_or]
    dsimp only at h
    split_ifs at h with ha
    · simp only [Option.some.injEq] at h
      subst h
      rw [stk_push_len]
      split_ifs <;> omega
    · cases ht : (s.stk (m % 2 == 1)).getLast? with
      | none => rw [ht] at h; simp at h
      | some p =>
        rw [ht] at h
        simp only at h
        split_ifs at h
        simp only [Option.some.injEq] at h
        subst h
        rw [length_setRef, stk_pop_len]
        split_ifs <;> omega

In [119]:
theorem cap_succ (m k : ℕ) (hk : k ≤ m) (σ : Bool) :
    cap m k σ = cap m (k + 1) σ + (if k < m ∨ (m % 2 == 1) = σ then 1 else 0) := by
  unfold cap
  rcases Nat.lt_or_ge k m with h | h
  · simp only [hk, ↓reduceIte, show k + 1 ≤ m by omega, h, true_or]
    split_ifs <;> omega
  · have : k = m := by omega
    subst this
    simp only [le_refl, ↓reduceIte, Nat.sub_self, zero_add, show ¬ k + 1 ≤ k by omega,
      lt_irrefl, false_or]
    cases σ <;> cases (k % 2 == 1) <;> simp

`viable_of_cnt` says
that a state with at least one accepted continuation is viable. If a sum is not zero, then some
term is not zero (`List.exists_mem_ne_zero_of_sum_ne_zero`), and that term is a successor with
an accepted continuation.

In [120]:
/-- A state with accepted continuations is viable. -/
theorem viable_of_cnt (m : ℕ) : ∀ (f k : ℕ) (s : St), f + k = m + 1 →
    cntFrom m f k s ≠ 0 → viable m k s = true := by
  intro f
  induction f with
  | zero =>
    intro k s hk hc
    rw [cntFrom_zero] at hc
    split_ifs at hc with hs
    · subst hs
      simp [viable, cap, final, St.stk, show ¬ k ≤ m by omega]
    · exact absurd rfl hc
  | succ f ih =>
    intro k s hk hc
    rw [cntFrom_succ] at hc
    obtain ⟨n, hn, hn0⟩ := List.exists_mem_ne_zero_of_sum_ne_zero hc
    obtain ⟨a, _, rfl⟩ := List.mem_map.1 hn
    cases hst : step m k s a with
    | none => rw [hst] at hn0; exact absurd rfl hn0
    | some s' =>
      rw [hst] at hn0
      have hv := ih (k + 1) s' (by omega) hn0
      simp only [viable, Bool.and_eq_true, decide_eq_true_eq] at hv ⊢
      have h1 := step_len hst true
      have h2 := step_len hst false
      rw [cap_succ m k (by omega) true, cap_succ m k (by omega) false]
      omega

**`by_contra`.** `by_contra h` proves a goal `P` by assuming `h : ¬ P` and deriving a
contradiction. Dropping non-viable states changes no weighted sum, because their weight is
zero.

In [121]:
theorem filter_viable_wsum (m k : ℕ) (hk : k ≤ m + 1) (L : List (St × ℕ)) :
    wsum (cntFrom m (m + 1 - k) k) (L.filter fun sc => viable m k sc.1) =
      wsum (cntFrom m (m + 1 - k) k) L := by
  induction L with
  | nil => rfl
  | cons sc L ih =>
    by_cases hv : viable m k sc.1 = true
    · simp only [List.filter_cons, hv, ↓reduceIte, wsum, List.map_cons, List.sum_cons] at ih ⊢
      rw [ih]
    · have h0 : cntFrom m (m + 1 - k) k sc.1 = 0 := by
        by_contra hc; exact hv (viable_of_cnt m _ k sc.1 (by omega) hc)
      simp only [List.filter_cons, hv, Bool.false_eq_true, ↓reduceIte, wsum, List.map_cons,
        List.sum_cons, h0, mul_zero, zero_add] at ih ⊢
      exact ih

In [122]:
theorem insertAdd_wsum (g : St → ℕ) (sc : St × ℕ) (L : List (St × ℕ)) :
    wsum g (insertAdd sc L) = sc.2 * g sc.1 + wsum g L := by
  induction L with
  | nil => simp [insertAdd, wsum]
  | cons tc L ih =>
    obtain ⟨t, c⟩ := tc
    by_cases h : t = sc.1
    · simp only [insertAdd, h, ↓reduceIte, wsum, List.map_cons, List.sum_cons]; ring
    · simp only [insertAdd, h, ↓reduceIte, wsum, List.map_cons, List.sum_cons] at ih ⊢
      rw [ih]; ring

In [123]:
theorem compressK_wsum (g : St → ℕ) (L : List (St × ℕ)) : wsum g (compressK L) = wsum g L := by
  induction L with
  | nil => rfl
  | cons sc L ih =>
    simp only [compressK, List.foldr_cons] at ih ⊢
    rw [insertAdd_wsum, ih]
    simp [wsum]

### The count

**Hypotheses about functions.** `layer_wsum` takes any merge `comp` together with a proof
`hcomp` that it keeps all weighted sums, `∀ g L, wsum g (comp L) = wsum g L`. By induction on
the number of points processed, the weighted count of accepted continuations of the current
layer never changes, so it equals the total for the initial state.

In [124]:
/-- After `k` points, the layer's weighted count of accepted continuations is the total. -/
theorem layer_wsum (comp : List (St × ℕ) → List (St × ℕ))
    (hcomp : ∀ g L, wsum g (comp L) = wsum g L) (m : ℕ) : ∀ k ≤ m + 1,
    wsum (cntFrom m (m + 1 - k) k) (layerWith comp m k) = cntFrom m (m + 1) 0 init := by
  intro k
  induction k with
  | zero => intro _; simp [layerWith, wsum]
  | succ k ih =>
    intro hk
    rw [layerWith, filter_viable_wsum m (k + 1) hk, hcomp,
      show m + 1 - (k + 1) = m - k by omega, expand_wsum,
      show m - k + 1 = m + 1 - k by omega, ih (by omega)]

In [125]:
/-- **The transfer matrix counts the accepted action sequences**, with any merge step that
keeps weighted sums. -/
theorem tmCountWith_eq_countP (comp : List (St × ℕ) → List (St × ℕ))
    (hcomp : ∀ g L, wsum g (comp L) = wsum g L) (m : ℕ) :
    tmCountWith comp m =
      (sufWords m (m + 1) 0).countP fun w => runFrom m 0 init w = some final := by
  have h := layer_wsum comp hcomp m (m + 1) le_rfl
  rw [Nat.sub_self] at h
  unfold tmCountWith
  rw [← cntFrom, ← h]
  unfold wsum
  congr 1
  apply List.map_congr_left
  intro sc _
  rw [cntFrom_zero]
  split <;> simp

What remains is the hard part: accepted action sequences correspond one to one with meanders.
That proof is 2,500 lines across eight files, and the next section uses it from the compiled
library.

## 12. Using the whole library

The rest of the formalization is about 3,000 lines: the correspondence between accepted action
sequences and meanders, the bound by Catalan numbers, and the certified values. Rather than
replay it, we import the compiled library. A cell that begins with `import` starts a new file,
so everything above is set aside, and we refer to the library's names with their full prefix.

In [126]:
import Arnold

### Accepted sequences are meanders

To prove that the transfer matrix counts meanders, the library runs a richer *decorated*
machine alongside it. Each open arc also remembers the actual piece of river it leads into, and
forgetting the decorations gives back the plain machine. Two maps connect the two sides:

* `encode m l` is the action sequence that a river with crossing order `l` induces: each point
  opens its arc on a side exactly when the other end of that arc lies to the east.
* `decode m w` runs the decorated machine on an accepted sequence `w` and reads off the one
  remaining piece, which is the whole river.

`#check` prints the statements. `decode_spec` says that decoding an accepted sequence gives a
meander that encodes back to the same sequence. `encode_spec` says that a meander's sequence is
accepted and decodes back to the meander. Together they are a bijection, and the main theorem
follows. The `@` in `#check @f` shows the implicit arguments too.

In [127]:
#check @Arnold.TM.decode_spec
#check @Arnold.TM.encode_spec
#check @Arnold.TM.tmCountWith_eq
#check Arnold.TM.tmCount_eq

@Arnold.TM.decode_spec : ∀ {m : ℕ} {w : List (Bool × Bool)},
  w ∈ Arnold.TM.sufWords m (m + 1) 0 →
    Arnold.TM.runFrom m 0 Arnold.TM.init w = some Arnold.TM.final →
      (Arnold.TM.decode m w).Perm (List.range m) ∧
        Arnold.NoCross (Arnold.lpt m (Arnold.TM.decode m w)) (m + 1) ∧ Arnold.TM.encode m (Arnold.TM.decode m w) = w


@Arnold.TM.encode_spec : ∀ {m : ℕ} {l : List ℕ},
  l.Perm (List.range m) →
    Arnold.NoCross (Arnold.lpt m l) (m + 1) →
      Arnold.TM.encode m l ∈ Arnold.TM.sufWords m (m + 1) 0 ∧
        Arnold.TM.runFrom m 0 Arnold.TM.init (Arnold.TM.encode m l) = some Arnold.TM.final ∧
          Arnold.TM.decode m (Arnold.TM.encode m l) = l


Arnold.TM.tmCountWith_eq : ∀ (comp : List (Arnold.TM.St × ℕ) → List (Arnold.TM.St × ℕ)),
  (∀ (g : Arnold.TM.St → ℕ) (L : List (Arnold.TM.St × ℕ)), Arnold.TM.wsum g (comp L) = Arnold.TM.wsum g L) →
    ∀ (m : ℕ), Arnold.TM.tmCountWith comp m = Arnold.openMeanderCount m


Arnold.TM.tmCount_eq (m : ℕ) : Arnold.TM.tmCount m = Arnold.openMeanderCount m


In [128]:
#print axioms Arnold.TM.tmCount_eq

'Arnold.TM.tmCount_eq' depends on axioms: [propext, Classical.choice, Quot.sound]


### Values checked by the kernel, and values computed by the compiler

**`decide +kernel`.** Plain `decide` first evaluates the decision procedure in the elaborator,
then has the kernel check the result. `decide +kernel` skips the first step and asks the kernel
alone, which is faster for large computations. `tmCountK` uses the merge by insertion, which the
kernel evaluates well. The library proves the first fourteen values this way
(`Arnold.openMeanderCount_values`); here is one of them, from scratch.

In [129]:
example : Arnold.openMeanderCount 11 = 1828 := by
  rw [← Arnold.TM.tmCountK_eq]
  decide +kernel

**`native_decide`.** For larger values the library compiles the decision procedure to machine
code and runs it. That trusts the compiler, and Lean records the trust as an extra axiom made
for the occasion, named after the theorem (here `arnold_20._native.native_decide.ax_1`), which
`#print axioms` reveals. The library's style linter flags
`native_decide`, so the theorem switches the linter off with `set_option … in`.

In [130]:
set_option linter.style.native false in
theorem arnold_20 : Arnold.openMeanderCount 20 = 19304190 := by
  rw [← Arnold.TM.tmCount_eq]
  native_decide

#print axioms arnold_20

'arnold_20' depends on axioms: [propext, Classical.choice, Quot.sound, arnold_20._native.native_decide.ax_1]


In [131]:
#check Arnold.openMeanderCount_values
#check Arnold.closedMeanderCount_values
#check Arnold.openMeanderCount_values_native

Arnold.openMeanderCount_values :
  List.map Arnold.openMeanderCount (List.range 14) = [1, 1, 1, 2, 3, 8, 14, 42, 81, 262, 538, 1828, 3926, 13820]


Arnold.closedMeanderCount_values :
  List.map Arnold.closedMeanderCount (List.range' 1 7) = [1, 2, 8, 42, 262, 1828, 13820]


Arnold.openMeanderCount_values_native :
  List.map Arnold.openMeanderCount (List.range 25) =
    [1, 1, 1, 2, 3, 8, 14, 42, 81, 262, 538, 1828, 3926, 13820, 30694, 110954, 252939, 933458, 2172830, 8152860,
      19304190, 73424650, 176343390, 678390116, 1649008456]


### The Catalan bound

A closed meander is two non-crossing perfect matchings of its $2n$ bridges glued together, the
arcs above the road and the arcs below. A non-crossing matching is determined by its Dyck word,
and there are $C_n$ Dyck words of length $2n$, so there are at most $C_n^2$ closed meanders. The
library proves this in `Arnold/Catalan.lean`, with matchings as a structure whose fields include
proofs: the partner map, and proofs that it is an involution without fixed points that never
crosses.

In [132]:
#check @Arnold.NCMatching.nc
#check @Arnold.closedMeanderCount_le_catalan_sq

@Arnold.NCMatching.nc : ∀ {N : ℕ} (self : Arnold.NCMatching N),
  ∀ a < N, ∀ b < N, ¬Arnold.Interleave a (self.m a) b (self.m b)


Arnold.closedMeanderCount_le_catalan_sq : ∀ (n : ℕ), Arnold.closedMeanderCount n ≤ catalan n ^ 2


The bound is far from tight. `List.range' 1 8` is the list `[1, …, 8]`, and Mathlib's
`catalan n` is $C_n$. The closed count uses $\mathrm{closed}(n) = \mathrm{open}(2n-1)$ from
Section 6.

In [133]:
#eval (List.range' 1 8).map fun n => (n, Arnold.TM.tmCount (2 * n - 1), catalan n ^ 2)

[(1, 1, 1), (2, 2, 4), (3, 8, 25), (4, 42, 196), (5, 262, 1764), (6, 1828, 17424), (7, 13820, 184041),
  (8, 110954, 2044900)]


### The `meanders` program

**Programs and `IO`.** A Lean program is a function `main` of type `IO Unit`, or `List String →
IO Unit` to receive command-line arguments. A value of type `IO α` is an action that interacts
with the world and returns an `α`. Actions are combined with `do`: each line is an action, `let
x := e` names a value, and `for x in l do` repeats an action. `IO.println` prints a line, and
`s!"…{e}…"` builds a string with the value of `e` inserted. `args.head?` is the first argument
as an `Option`, `String.toNat?` parses a number (also an `Option`), `o >>= f` chains two steps
that may fail like `Option.bind`, and `o.getD d` takes the value or the default `d`.

This is `Main.lean`, the program `lake exe meanders`, which prints Arnold's numbers with the
transfer matrix. `#eval` can run an `IO` action directly.

In [134]:
/-- Print the number of ways a river can cross a road `n` times, for `n = 0, …, N`
(default `N = 30`). Each value is computed by `Arnold.TM.tmCount`, the transfer matrix, which
`Arnold.TM.tmCount_eq` proves equal to `Arnold.openMeanderCount`. -/
def main (args : List String) : IO Unit := do
  let N := (args.head? >>= String.toNat?).getD 30
  for n in List.range (N + 1) do
    IO.println s!"{n}\t{Arnold.TM.tmCount n}"

In [135]:
#eval main ["24"]

0	1
1	1
2	1
3	2
4	3
5	8
6	14
7	42
8	81
9	262
10	538
11	1828
12	3926
13	13820
14	30694
15	110954
16	252939
17	933458
18	2172830
19	8152860
20	19304190
21	73424650
22	176343390
23	678390116
24	1649008456


### Beyond the proofs

The repository also has two programs that run the same machine much faster, without proofs:
`rust/` in Rust and `oxcaml/` in OxCaml. They pack a state into a single 64-bit word. Reading
the open arcs along the cut from top to bottom, the pieces west of the cut never cross, so the
arcs form a non-crossing matching, which is a bracket word. Both programs spread each layer
over all cores. Their counts agree with `tmCount` wherever both have run, and with the published
values in the OEIS. They are checked, not proved: the Lean values are the reference.

## 13. Index of Lean features

Each feature is explained in the section listed, before its first use.

| Feature | Section |
|---|---|
| `import`, Mathlib | 1 |
| `#eval`, `#check`, `--` comments | 1 |
| `Nat`, `ℕ`, type ascription `(e : T)`, function application | 1 |
| truncated `-`, `/`, `%`, pairs `(a, b)` and `A × B` | 1 |
| `Prop`, propositions as types | 1 |
| `namespace`, `end` | 2, 10 |
| `def`, docstrings `/-- … -/` | 2 |
| `¬`, `∧`, `↔`, `<`, `min`, `max` | 2 |
| type classes, `instance`, `Decidable` | 2 |
| tactic blocks `by`, `unfold`, `infer_instance`, `;` | 2 |
| function types `A → B`, `∀ x < n,`, implication `→` | 2 |
| `fun x => e`, lists `[a, b]`, `List ℕ`, dot notation | 2 |
| `Fin n`, `Equiv.Perm` | 3 |
| implicit arguments `{x : T}` | 3 |
| anonymous constructor `⟨a, b⟩`, dependent `if h : c` | 3 |
| coercions `↑x` | 3 |
| `List.range`, `List.map`, partial application | 3 |
| subtypes `{x // P x}`, `Fintype.card` | 3 |
| `theorem`, `example` | 4 |
| goals; `intro`, `rw`, `omega`, `≠` | 4 |
| `split_ifs`, `by_cases`, `·` | 4 |
| `simp`, `simp only`, simprocs `↓reduceIte`, `show … by`, `rfl` | 4 |
| `have`, `(by tac)` as an argument | 4 |
| term-mode proofs, `.mp`, `.mpr`, `Nonempty`, `_` | 4 |
| `calc` | 4 |
| `decide` | 4 |
| lemmas as functions, e.g. `Nat.mod_lt` | 5 |
| `∀ h : P,` over proofs | 5 |
| `Equiv`, `≃`, `.symm`, `.trans`, `Fin.revPerm`, `Perm.decomposeFin` | 6 |
| `open` | 6 |
| proofs of `∀` as functions; congruence lemmas | 6 |
| `∨`, `have :=` and `this`, `.isLt` | 6 |
| named arguments `(n := e)`, `↓reduceDIte`, `ext`, `rcases … with h \| h`, `<;>` | 6 |
| `refine` with holes `?_`, `generalize` | 6 |
| `exact`, `Function.Injective`, `congrArg`, `simpa`, `Equiv.ext` | 6 |
| `∃`, `obtain`, the pattern `rfl`, `subst`, `rw … at h`, `congr` | 6 |
| `Fintype.card_of_bijective`, `Subtype.ext`, `.1`, `.2`, `fun _ =>`, `symm`, `rintro` | 6 |
| `congr 1` | 6 |
| `Bool`, `decide`, `b = true` | 7 |
| `List.countP` | 7 |
| definitions by pattern matching, structural recursion | 7 |
| `++`, `[x]`, `List.erase`, `List.sum` | 7 |
| `&&`, `\|\|`, `!`, `==`, `!=` | 8 |
| `induction … with`, `constructor` | 8 |
| `Array`, `let` | 8 |
| `split`, `simp_all`, `Or.inl`, `Or.inr`, `.resolve_left`, `show … from`, `funext` | 8 |
| attributes `@[…]`, `@[csimp]`, `@f`, `apply` | 8 |
| `::`, `flatMap`, `(f ·)` | 9 |
| `∈`, `List.Perm`, `cases … with`, `simp … at h` | 9 |
| `Nodup`, `fun {x} =>`, `.symm` on equations | 9 |
| `∑ i ∈ s, f i`, `Finset`, `suffices`, `Nat.le_induction` | 9 |
| recursive proofs | 9 |
| `rw [← h]` | 9 |
| `obtain … : T := …`, `swap` | 9 |
| `List.ofFn`, `congrFun`, `l[i]'h` | 9 |
| instance arguments `[C α]`, `DecidablePred`, `Finset.card_bij` | 9 |
| `absurd` | 9 |
| `#print axioms` | 9 |
| `inductive`, constructors, `deriving` | 10 |
| `structure`, fields, `{ s with … }` | 10 |
| `Option`, `match` | 10 |
| matching several values, literal patterns | 10 |
| automatic `decide` from `Prop` to `Bool` | 10 |
| well-founded recursion | 10 |
| `filterMap`, `Option.map`, `filter`, `mergeSort`, `foldr` | 10 |
| functions as parameters | 10 |
| `set_option`, `Option.bind` | 11 |
| `cases h : e with` | 11 |
| functional induction `f.induct`, `ring`, `at h ⊢` | 11 |
| `rcases` on Booleans, `split_ifs at`, `dsimp`, `reduceCtorEq`, `if_pos` | 11 |
| `by_contra` | 11 |
| `decide +kernel`, `native_decide` and its axiom | 12 |
| structures with proof fields | 12 |
| `IO`, `do`, `for`, `s!"…"`, `>>=`, `main` | 12 |